In [10]:

# Cell 1: Check your Colab environment

import torch
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU for now.")


Python version: 3.13.16
PyTorch version: 2.11.0+cpu
GPU available: False
Running on CPU for now.


In [11]:

# Cell 2: Install the AI model libraries

%pip install -q -U transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.6 MB/s eta 0:00:00


In [12]:

# Cell 3: Load the model

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")
print("Device:", model.device)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Device: cpu


In [13]:

# Cell 4: User query understanding

import json

def understand_query(user_query):
    messages = [
        {
            "role": "system",
            "content": """
You extract structured information for a GeoAI outdoor
activity recommendation assistant.

Return ONLY a valid JSON object with these fields:
activity, location, radius_km, date, time_range,
health_conditions, age_group.

Rules:
- Use null for missing or uncertain information.
- Never invent a location, date, or personal detail.
- Keep the user's original date/time wording if exact
  dates or times cannot be determined.
- health_conditions must be a list.
- radius_km must be a number or null.
- Do not give the recommendation yet.
"""
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    # Extract JSON safely
    start = answer.find("{")
    end = answer.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"Model did not return JSON: {answer}")

    return json.loads(answer[start:end + 1])


# Test your AI brain
query = (
    "Where can I go running tomorrow morning "
    "within 5 km of Powai? I have mild asthma."
)

result = understand_query(query)
print(json.dumps(result, indent=2))


{
  "activity": "running",
  "location": "Powai",
  "radius_km": 5,
  "date": "tomorrow",
  "time_range": {
    "start_time": "morning",
    "end_time": ""
  },
  "health_conditions": [
    "mild asthma"
  ],
  "age_group": null
}


In [14]:

# Cell 5: Updated scoring engine with asthma awareness

def score_conditions(
    activity,
    weather,
    air_quality,
    health_conditions=None
):
    health_conditions = health_conditions or []

    health_text = " ".join(
        str(condition).lower()
        for condition in health_conditions
    )

    score = 100
    warnings = []
    missing_data = []

    temp = weather.get("temperature_c")
    rain = weather.get("precipitation_probability")
    wind = weather.get("wind_speed_kmh")
    pm25 = air_quality.get("pm2_5")

    for name, value in [
        ("temperature", temp),
        ("precipitation", rain),
        ("wind speed", wind),
        ("PM2.5", pm25)
    ]:
        if value is None:
            missing_data.append(name)

    if temp is not None:
        if temp >= 36:
            score -= 25
            warnings.append("Very hot conditions.")
        elif temp >= 30:
            score -= 10
            warnings.append("Warm conditions: consider heat exposure.")

    if rain is not None:
        if rain >= 70:
            score -= 25
            warnings.append("High precipitation probability.")
        elif rain >= 40:
            score -= 15
            warnings.append("Rain may disrupt the activity.")
        elif rain >= 20:
            score -= 5
            warnings.append("Check the precipitation forecast.")

    if wind is not None:
        if wind >= 35:
            score -= 20
            warnings.append("Strong winds may affect outdoor safety.")
        elif wind >= 20:
            score -= 10
            warnings.append("Wind may affect activity comfort.")

    if pm25 is not None:
        if pm25 < 0:
            missing_data.append("valid PM2.5 measurement")
            warnings.append("Invalid PM2.5 measurement.")
        elif pm25 >= 60:
            score -= 35
            warnings.append("Elevated PM2.5: reassess exposure.")
        elif pm25 >= 35:
            score -= 20
            warnings.append("Increased PM2.5 requires caution.")
        elif pm25 >= 15:
            score -= 10
            warnings.append("Consider air-quality conditions.")

    if "asthma" in health_text:
        if pm25 is None:
            warnings.append(
                "Asthma reported: air-quality data is unavailable."
            )
        elif pm25 >= 15:
            warnings.append(
                "Asthma reported: review air quality and personal advice."
            )

    return {
        "score": max(0, min(100, score)),
        "warnings": warnings,
        "missing_data": sorted(set(missing_data))
    }

print("Updated scoring function loaded successfully!")


Updated scoring function loaded successfully!


In [15]:
# Cell 6: Updated comparison engine
# Supports health conditions and ranks multiple candidates.

def compare_candidates(candidates, activity, health_conditions=None):

    if health_conditions is None:
        health_conditions = []

    results = []

    for candidate in candidates:

        assessment = score_conditions(
            activity=activity,
            weather=candidate.get("weather", {}),
            air_quality=candidate.get("air_quality", {}),
            health_conditions=health_conditions
        )

        results.append({
            "place": candidate.get("place", "Unknown"),
            "distance_km": candidate.get("distance_km"),
            "time": candidate.get("time"),
            "score": assessment["score"],
            "warnings": assessment["warnings"],
            "missing_data": assessment["missing_data"],
            "weather": candidate.get("weather", {}),
            "air_quality": candidate.get("air_quality", {}),
            "geospatial_features": candidate.get(
                "geospatial_features", {}
            )
        })

    # Rank candidates by data completeness and score.
    results.sort(
        key=lambda item: (
            len(item["missing_data"]) == 0,
            -len(item["missing_data"]),
            item["score"]
        ),
        reverse=True
    )

    return results


print("Updated comparison engine loaded successfully!")

Updated comparison engine loaded successfully!


In [17]:
# Cell 7: Test the updated comparison engine

ranked = compare_candidates(
    candidates=demo_candidates,
    activity="running",
    health_conditions=["mild asthma"]
)

for item in ranked:
    print(f"Place: {item['place']}")
    print(f"Time: {item['time']}")
    print(f"Score: {item['score']}/100")
    print(f"Warnings: {item['warnings']}")
    print(f"Missing data: {item['missing_data']}")
    print("-" * 40)

NameError: name 'demo_candidates' is not defined

In [18]:

import requests

# Test the real geocoding API using our parsed example location
place_name = "Powai, Mumbai, India"

url = "https://geocoding-api.open-meteo.com/v1/search"
params = {
    "name": place_name,
    "count": 5,
    "language": "en",
    "format": "json"
}

response = requests.get(url, params=params, timeout=20)
response.raise_for_status()
data = response.json()

if data.get("results"):
    for place in data["results"]:
        print("Place:", place["name"])
        print("Country:", place.get("country"))
        print("Latitude:", place["latitude"])
        print("Longitude:", place["longitude"])
        print("Timezone:", place.get("timezone"))
        print("-" * 35)
else:
    print("No matching location found.")


No matching location found.


In [19]:
import requests

url = "https://geocoding-api.open-meteo.com/v1/search"

params = {
    "name": "Mumbai",
    "count": 5,
    "language": "en",
    "format": "json"
}

response = requests.get(url, params=params, timeout=20)

print("HTTP Status:", response.status_code)
response.raise_for_status()

data = response.json()

if data.get("results"):
    for place in data["results"]:
        print("Place:", place["name"])
        print("State:", place.get("admin1"))
        print("Country:", place.get("country"))
        print("Latitude:", place["latitude"])
        print("Longitude:", place["longitude"])
        print("Timezone:", place.get("timezone"))
        print("-" * 35)
else:
    print("No results returned.")
    print("API response:", data)


HTTP Status: 200
Place: Mumbai
State: Maharashtra
Country: India
Latitude: 19.07283
Longitude: 72.88261
Timezone: Asia/Kolkata
-----------------------------------


In [20]:

import requests
from datetime import datetime
from zoneinfo import ZoneInfo

# Coordinates returned by our successful geocoding test
latitude = 19.07283
longitude = 72.88261
timezone = "Asia/Kolkata"

# Demo comparison time — change this to test another hour
target_hour = "07:00"

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": [
        "temperature_2m",
        "apparent_temperature",
        "precipitation_probability",
        "precipitation",
        "relative_humidity_2m",
        "wind_speed_10m",
        "weather_code"
    ],
    "forecast_days": 3,
    "timezone": timezone
}

response = requests.get(url, params=params, timeout=30)
print("HTTP Status:", response.status_code)
response.raise_for_status()

weather = response.json()
hourly = weather["hourly"]

# Select the next future occurrence of the requested local hour
tz = ZoneInfo(timezone)
now = datetime.now(tz)

matches = [
    i for i, timestamp in enumerate(hourly["time"])
    if timestamp.endswith("T" + target_hour)
    and datetime.fromisoformat(timestamp).replace(tzinfo=tz) > now
]

if not matches:
    print("No future forecast found for", target_hour)
else:
    i = matches[0]

    print("\nREAL WEATHER FORECAST")
    print("Location: Mumbai, Maharashtra")
    print("Forecast time:", hourly["time"][i])

    fields = {
        "Temperature (°C)": "temperature_2m",
        "Feels like (°C)": "apparent_temperature",
        "Precipitation probability (%)": "precipitation_probability",
        "Precipitation (mm)": "precipitation",
        "Relative humidity (%)": "relative_humidity_2m",
        "Wind speed (km/h)": "wind_speed_10m",
        "Weather code": "weather_code"
    }

    for label, key in fields.items():
        print(f"{label}: {hourly[key][i]}")


HTTP Status: 200

REAL WEATHER FORECAST
Location: Mumbai, Maharashtra
Forecast time: 2026-10-23T07:00
Temperature (°C): 27.8
Feels like (°C): 30.6
Precipitation probability (%): 0
Precipitation (mm): 0.0
Relative humidity (%): 62
Wind speed (km/h): 6.8
Weather code: 0


In [21]:

import requests
from datetime import datetime
from zoneinfo import ZoneInfo

latitude = 19.07283
longitude = 72.88261
timezone = "Asia/Kolkata"
target_hour = "07:00"

url = "https://air-quality-api.open-meteo.com/v1/air-quality"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": [
        "pm2_5",
        "pm10",
        "european_aqi",
        "us_aqi",
        "ozone",
        "nitrogen_dioxide"
    ],
    "forecast_days": 3,
    "timezone": timezone
}

response = requests.get(url, params=params, timeout=30)

print("HTTP Status:", response.status_code)
response.raise_for_status()

air_data = response.json()
air_hourly = air_data["hourly"]

# Find the next future occurrence of the requested hour
tz = ZoneInfo(timezone)
now = datetime.now(tz)

matches = [
    i for i, timestamp in enumerate(air_hourly["time"])
    if timestamp.endswith("T" + target_hour)
    and datetime.fromisoformat(timestamp).replace(tzinfo=tz) > now
]

if not matches:
    print("No future air-quality forecast found for", target_hour)
else:
    i = matches[0]

    print("\nREAL AIR-QUALITY FORECAST")
    print("Location: Mumbai, Maharashtra")
    print("Forecast time:", air_hourly["time"][i])

    fields = {
        "PM2.5 (µg/m³)": "pm2_5",
        "PM10 (µg/m³)": "pm10",
        "European AQI": "european_aqi",
        "US AQI": "us_aqi",
        "Ozone (µg/m³)": "ozone",
        "Nitrogen dioxide (µg/m³)": "nitrogen_dioxide"
    }

    for label, key in fields.items():
        value = air_hourly.get(key, [None] * len(air_hourly["time"]))[i]
        print(f"{label}: {value}")


HTTP Status: 200

REAL AIR-QUALITY FORECAST
Location: Mumbai, Maharashtra
Forecast time: 2026-10-23T07:00
PM2.5 (µg/m³): 58.3
PM10 (µg/m³): 62.6
European AQI: 64
US AQI: 160
Ozone (µg/m³): 51.0
Nitrogen dioxide (µg/m³): 32.4


In [22]:

# Step 4: Combine real weather and air-quality data

weather_hourly = weather["hourly"]
air_hourly = air_data["hourly"]

# Match records by their exact local forecast timestamp
weather_times = {
    timestamp: i
    for i, timestamp in enumerate(weather_hourly["time"])
}
air_times = {
    timestamp: i
    for i, timestamp in enumerate(air_hourly["time"])
}

common_times = sorted(set(weather_times) & set(air_times))

if not common_times:
    raise ValueError(
        "No matching forecast timestamps between the APIs."
    )

# Use the next forecast time that both APIs provide
tz = ZoneInfo(timezone)
now = datetime.now(tz)

future_times = [
    timestamp for timestamp in common_times
    if datetime.fromisoformat(timestamp).replace(tzinfo=tz) > now
]

if not future_times:
    raise ValueError("No matching future forecast timestamps.")

forecast_time = future_times[0]
wi = weather_times[forecast_time]
ai = air_times[forecast_time]

geo_record = {
    "location": "Mumbai, Maharashtra, India",
    "latitude": latitude,
    "longitude": longitude,
    "timezone": timezone,
    "forecast_time": forecast_time,

    "temperature_c": weather_hourly["temperature_2m"][wi],
    "feels_like_c": weather_hourly["apparent_temperature"][wi],
    "precipitation_probability_pct":
        weather_hourly["precipitation_probability"][wi],
    "precipitation_mm": weather_hourly["precipitation"][wi],
    "humidity_pct": weather_hourly["relative_humidity_2m"][wi],
    "wind_speed_kmh": weather_hourly["wind_speed_10m"][wi],
    "weather_code": weather_hourly["weather_code"][wi],

    "pm2_5": air_hourly["pm2_5"][ai],
    "pm10": air_hourly["pm10"][ai],
    "us_aqi": air_hourly["us_aqi"][ai],
    "european_aqi": air_hourly["european_aqi"][ai],
    "ozone": air_hourly["ozone"][ai],
    "nitrogen_dioxide": air_hourly["nitrogen_dioxide"][ai],
}

print("COMBINED GEO AI RECORD")
for key, value in geo_record.items():
    print(f"{key}: {value}")


COMBINED GEO AI RECORD
location: Mumbai, Maharashtra, India
latitude: 19.07283
longitude: 72.88261
timezone: Asia/Kolkata
forecast_time: 2026-10-22T15:00
temperature_c: 34.6
feels_like_c: 36.2
precipitation_probability_pct: 0
precipitation_mm: 0.0
humidity_pct: 34
wind_speed_kmh: 4.4
weather_code: 3
pm2_5: 103.4
pm10: 107.3
us_aqi: 232
european_aqi: 395
ozone: 475.0
nitrogen_dioxide: 47.6


In [23]:

# Step 5A: Run the existing scoring engine with real API data

# Map the combined API record into the function's expected inputs
weather_input = {
    "temperature_c": geo_record.get("temperature_c"),
    "precipitation_probability": geo_record.get(
        "precipitation_probability_pct"
    ),
    "wind_speed_kmh": geo_record.get("wind_speed_kmh")
}

air_quality_input = {
    "pm2_5": geo_record.get("pm2_5")
}

# Demo inputs; replace with the user's actual activity and conditions
activity = "outdoor walking"
health_conditions = ["asthma"]

result = score_conditions(
    activity=activity,
    weather=weather_input,
    air_quality=air_quality_input,
    health_conditions=health_conditions
)

print("GEO AI REAL-DATA SCORING RESULT")
print("Location:", geo_record["location"])
print("Forecast time:", geo_record["forecast_time"])
print("Activity:", activity)
print("Score:", f'{result["score"]}/100')
print("Warnings:", result["warnings"])
print("Missing data:", result["missing_data"])


GEO AI REAL-DATA SCORING RESULT
Location: Mumbai, Maharashtra, India
Forecast time: 2026-10-22T15:00
Activity: outdoor walking
Score: 55/100
Warnings: ['Warm conditions: consider heat exposure.', 'Elevated PM2.5: reassess exposure.', 'Asthma reported: review air quality and personal advice.']
Missing data: []


In [24]:

# Step 6: Validate the combined real-data record

required_fields = [
    "temperature_c",
    "precipitation_probability_pct",
    "wind_speed_kmh",
    "pm2_5",
    "pm10",
    "us_aqi",
    "forecast_time"
]

validation_issues = []

for field in required_fields:
    value = geo_record.get(field)

    if value is None:
        validation_issues.append(f"Missing data: {field}")

# Basic physical/range checks
checks = {
    "temperature_c": lambda x: -90 <= x <= 65,
    "precipitation_probability_pct": lambda x: 0 <= x <= 100,
    "wind_speed_kmh": lambda x: x >= 0,
    "pm2_5": lambda x: x >= 0,
    "pm10": lambda x: x >= 0,
    "us_aqi": lambda x: x >= 0
}

for field, is_valid in checks.items():
    value = geo_record.get(field)

    if value is not None and not is_valid(value):
        validation_issues.append(f"Invalid value: {field}={value}")

print("DATA VALIDATION")
print("Location:", geo_record.get("location"))
print("Forecast time:", geo_record.get("forecast_time"))

if validation_issues:
    for issue in validation_issues:
        print("-", issue)
else:
    print("All required fields are present.")
    print("All basic range checks passed.")


DATA VALIDATION
Location: Mumbai, Maharashtra, India
Forecast time: 2026-10-22T15:00
All required fields are present.
All basic range checks passed.


In [25]:

# Step 7A: Compare multiple real locations

import requests
from datetime import datetime
from zoneinfo import ZoneInfo

CITIES = [
    "Pune",
    "Mumbai",
    "Hyderabad",
    "Bengaluru",
    "Kolkata",
    "Jaipur",
    "Ahmedabad",
    "Chandigarh",
    "Delhi"
]


# Must be a future hour available in both forecast APIs.
# Change this to the visit time you want to compare.
TARGET_TIME = "2026-10-22T07:00"
TIMEZONE = "Asia/Kolkata"

ACTIVITY = "outdoor walking"
HEALTH_CONDITIONS = ["asthma"]

session = requests.Session()


def get_coordinates(city):
    response = session.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={
            "name": city,
            "count": 5,
            "language": "en",
            "format": "json",
            "countryCode": "IN"
        },
        timeout=30
    )
    response.raise_for_status()

    results = response.json().get("results", [])

    if not results:
        raise ValueError(f"No location found for {city}")

    # Prefer an exact city-name match
    place = next(
        (p for p in results
         if p["name"].casefold() == city.casefold()),
        results[0]
    )

    return place


def get_hourly_data(url, latitude, longitude, fields):
    response = session.get(
        url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "hourly": fields,
            "forecast_days": 3,
            "timezone": TIMEZONE
        },
        timeout=30
    )
    response.raise_for_status()
    return response.json()["hourly"]


def compare_city(city):
    place = get_coordinates(city)
    lat = place["latitude"]
    lon = place["longitude"]

    weather = get_hourly_data(
        "https://api.open-meteo.com/v1/forecast",
        lat, lon,
        [
            "temperature_2m",
            "apparent_temperature",
            "precipitation_probability",
            "precipitation",
            "relative_humidity_2m",
            "wind_speed_10m",
            "weather_code"
        ]
    )

    air = get_hourly_data(
        "https://air-quality-api.open-meteo.com/v1/air-quality",
        lat, lon,
        [
            "pm2_5",
            "pm10",
            "us_aqi",
            "european_aqi",
            "ozone",
            "nitrogen_dioxide"
        ]
    )

    # Match the requested local forecast time exactly
    weather_index = {
        timestamp: i
        for i, timestamp in enumerate(weather["time"])
    }
    air_index = {
        timestamp: i
        for i, timestamp in enumerate(air["time"])
    }

    if TARGET_TIME not in weather_index:
        raise ValueError(
            f"Weather forecast unavailable for {city} at {TARGET_TIME}"
        )

    if TARGET_TIME not in air_index:
        raise ValueError(
            f"Air-quality forecast unavailable for {city} at {TARGET_TIME}"
        )

    wi = weather_index[TARGET_TIME]
    ai = air_index[TARGET_TIME]

    weather_input = {
        "temperature_c": weather["temperature_2m"][wi],
        "precipitation_probability":
            weather["precipitation_probability"][wi],
        "wind_speed_kmh": weather["wind_speed_10m"][wi]
    }

    air_input = {
        "pm2_5": air["pm2_5"][ai]
    }

    result = score_conditions(
        activity=ACTIVITY,
        weather=weather_input,
        air_quality=air_input,
        health_conditions=HEALTH_CONDITIONS
    )

    return {
        "location": place["name"],
        "state": place.get("admin1", "Unknown"),
        "latitude": lat,
        "longitude": lon,
        "forecast_time": TARGET_TIME,
        "temperature_c": weather_input["temperature_c"],
        "pm2_5": air_input["pm2_5"],
        "us_aqi": air["us_aqi"][ai],
        "score": result["score"],
        "warnings": result["warnings"],
        "missing_data": result["missing_data"]
    }


results = []

for city in CITIES:
    try:
        results.append(compare_city(city))
    except (requests.RequestException, ValueError, KeyError) as error:
        print(f"Could not compare {city}: {error}")

# Highest heuristic score first
results.sort(key=lambda item: item["score"], reverse=True)

print("\nGEO AI MULTI-LOCATION COMPARISON")
print("Activity:", ACTIVITY)
print("Requested time:", TARGET_TIME)
print("-" * 55)

for item in results:
    print(f"\nLocation: {item['location']}, {item['state']}")
    print(f"Score: {item['score']}/100")
    print(f"Temperature: {item['temperature_c']} °C")
    print(f"PM2.5: {item['pm2_5']} µg/m³")
    print(f"US AQI: {item['us_aqi']}")
    print("Warnings:", item["warnings"])
    print("Missing data:", item["missing_data"])
    print("-" * 55)

if results:
    print("\nHighest-scoring location under current rules:")
    print(results[0]["location"], "-", results[0]["score"], "/100")
else:
    print("\nNo locations could be compared successfully.")



GEO AI MULTI-LOCATION COMPARISON
Activity: outdoor walking
Requested time: 2026-10-22T07:00
-------------------------------------------------------

Location: Bengaluru, Karnataka
Score: 95/100
Temperature: 21.5 °C
PM2.5: 7.3 µg/m³
US AQI: 53
Warnings: ['Check the precipitation forecast.']
Missing data: []
-------------------------------------------------------

Location: Pune, Maharashtra
Score: 90/100
Temperature: 23.7 °C
PM2.5: 27.9 µg/m³
US AQI: 101
Warnings: ['Consider air-quality conditions.', 'Asthma reported: review air quality and personal advice.']
Missing data: []
-------------------------------------------------------

Location: Hyderabad, Telangana
Score: 80/100
Temperature: 24.7 °C
PM2.5: 54.1 µg/m³
US AQI: 122
Warnings: ['Increased PM2.5 requires caution.', 'Asthma reported: review air quality and personal advice.']
Missing data: []
-------------------------------------------------------

Location: Jaipur, Rajasthan
Score: 80/100
Temperature: 21.1 °C
PM2.5: 42.1 µg/m³
U

In [26]:

# Step 8A: Compare 9 locations across morning, afternoon, and evening
# Uses existing Cell 7A functions and scoring engine.
# Keeps the original `results` variable unchanged for the existing map.

TIME_SLOTS = {
    "Morning": "07:00",
    "Afternoon": "13:00",
    "Evening": "18:00"
}

# Use the same date as TARGET_TIME in Cell 7A
TARGET_DATE = TARGET_TIME.split("T")[0]

multi_time_results = []
unavailable = []

print("GEO AI: MULTI-TIME LOCATION COMPARISON")
print("Activity:", ACTIVITY)
print("Date:", TARGET_DATE)
print("Time slots:", ", ".join(
    f"{label} ({hour})" for label, hour in TIME_SLOTS.items()
))
print("=" * 65)

for city in CITIES:
    try:
        # Fetch coordinates once per city
        place = get_coordinates(city)
        lat = place["latitude"]
        lon = place["longitude"]

        # Fetch each API's hourly data once per city
        weather = get_hourly_data(
            "https://api.open-meteo.com/v1/forecast",
            lat, lon,
            [
                "temperature_2m",
                "apparent_temperature",
                "precipitation_probability",
                "precipitation",
                "relative_humidity_2m",
                "wind_speed_10m",
                "weather_code"
            ]
        )

        air = get_hourly_data(
            "https://air-quality-api.open-meteo.com/v1/air-quality",
            lat, lon,
            [
                "pm2_5",
                "pm10",
                "us_aqi",
                "european_aqi",
                "ozone",
                "nitrogen_dioxide"
            ]
        )

        # Build timestamp indexes for exact local-time matching
        weather_index = {
            timestamp: i
            for i, timestamp in enumerate(weather["time"])
        }
        air_index = {
            timestamp: i
            for i, timestamp in enumerate(air["time"])
        }

    except (requests.RequestException, ValueError, KeyError) as error:
        print(f"Could not fetch data for {city}: {error}")
        unavailable.append({
            "location": city,
            "time": "All slots",
            "reason": str(error)
        })
        continue

    # Score each requested time using the existing scoring engine
    for period, hour in TIME_SLOTS.items():
        target_time = f"{TARGET_DATE}T{hour}"

        if (target_time not in weather_index or
                target_time not in air_index):
            unavailable.append({
                "location": city,
                "time": f"{period} ({target_time})",
                "reason": "Timestamp missing from weather or air-quality forecast"
            })
            continue

        wi = weather_index[target_time]
        ai = air_index[target_time]

        try:
            weather_input = {
                "temperature_c": weather["temperature_2m"][wi],
                "precipitation_probability":
                    weather["precipitation_probability"][wi],
                "wind_speed_kmh": weather["wind_speed_10m"][wi]
            }

            air_input = {
                "pm2_5": air["pm2_5"][ai]
            }

            score_result = score_conditions(
                activity=ACTIVITY,
                weather=weather_input,
                air_quality=air_input,
                health_conditions=HEALTH_CONDITIONS
            )

            multi_time_results.append({
                "location": place["name"],
                "state": place.get("admin1", "Unknown"),
                "latitude": lat,
                "longitude": lon,
                "period": period,
                "forecast_time": target_time,
                "temperature_c": weather_input["temperature_c"],
                "pm2_5": air_input["pm2_5"],
                "us_aqi": air["us_aqi"][ai],
                "score": score_result["score"],
                "warnings": score_result["warnings"],
                "missing_data": score_result["missing_data"]
            })

        except (ValueError, KeyError, IndexError, TypeError) as error:
            unavailable.append({
                "location": city,
                "time": f"{period} ({target_time})",
                "reason": str(error)
            })

# Rank results without modifying the original `results` list
multi_time_results.sort(
    key=lambda item: item["score"],
    reverse=True
)

# Display rankings separately for each time of day
for period in TIME_SLOTS:
    period_results = [
        item for item in multi_time_results
        if item["period"] == period
    ]
    period_results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    print(f"\n{'-' * 65}")
    print(f"{period.upper()} — {TARGET_DATE} "
          f"({TIME_SLOTS[period]})")
    print("-" * 65)

    if not period_results:
        print("No locations returned usable results for this time.")
        continue

    for rank, item in enumerate(period_results, start=1):
        print(
            f"{rank}. {item['location']}, {item['state']} "
            f"| Score: {item['score']}/100"
        )
        print(
            f"   Temperature: {item['temperature_c']} °C"
            f" | PM2.5: {item['pm2_5']} µg/m³"
            f" | US AQI: {item['us_aqi']}"
        )
        print("   Warnings:", item["warnings"])
        print("   Missing data:", item["missing_data"])

# Best city-time combination under the current scoring rules
print(f"\n{'=' * 65}")
print("BEST AVAILABLE CITY-TIME COMBINATION")

if multi_time_results:
    best = multi_time_results[0]
    print(
        f"{best['location']}, {best['state']} — {best['period']}"
    )
    print("Forecast time:", best["forecast_time"])
    print("Score:", f"{best['score']}/100")
    print("Warnings:", best["warnings"])
else:
    print("No city-time combinations could be scored.")

# Report unavailable combinations rather than inventing results
print("\nDATA AVAILABILITY SUMMARY")
print("Successful city-time comparisons:", len(multi_time_results))
print("Unavailable or failed items:", len(unavailable))

for item in unavailable:
    print(
        f"- {item['location']} | {item['time']} | {item['reason']}"
    )


GEO AI: MULTI-TIME LOCATION COMPARISON
Activity: outdoor walking
Date: 2026-10-22
Time slots: Morning (07:00), Afternoon (13:00), Evening (18:00)

-----------------------------------------------------------------
MORNING — 2026-10-22 (07:00)
-----------------------------------------------------------------
1. Bengaluru, Karnataka | Score: 95/100
   Temperature: 21.5 °C | PM2.5: 7.3 µg/m³ | US AQI: 53
   Warnings: ['Check the precipitation forecast.']
   Missing data: []
2. Pune, Maharashtra | Score: 90/100
   Temperature: 23.7 °C | PM2.5: 27.9 µg/m³ | US AQI: 101
   Warnings: ['Consider air-quality conditions.', 'Asthma reported: review air quality and personal advice.']
   Missing data: []
3. Hyderabad, Telangana | Score: 80/100
   Temperature: 24.7 °C | PM2.5: 54.1 µg/m³ | US AQI: 122
   Warnings: ['Increased PM2.5 requires caution.', 'Asthma reported: review air quality and personal advice.']
   Missing data: []
4. Jaipur, Rajasthan | Score: 80/100
   Temperature: 21.1 °C | PM2.5: 4

In [28]:
# Step 9A: Check the environment for LLM integration

import torch
import transformers

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. We can consider a lighter CPU-compatible setup.")

PyTorch version: 2.11.0+cpu
Transformers version: 5.20.2
GPU available: False
No GPU detected. We can consider a lighter CPU-compatible setup.


In [27]:
# Step 9B: Check available system memory

import os

try:
    import psutil
except ImportError:
    %pip install -q psutil
    import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"CPU cores: {os.cpu_count()}")

Total RAM: 12.67 GB
Available RAM: 9.36 GB
CPU cores: 2


In [30]:
%pip install -q llama-cpp-python

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [34]:
# Step 9D: Verify llama.cpp installation

import llama_cpp

print("llama-cpp-python installed successfully!")
print("Version:", llama_cpp.__version__)
print("LLM integration library is ready.")

llama-cpp-python installed successfully!
Version: 0.3.36
LLM integration library is ready.


In [35]:
# Step 9E: Diagnose the installation

import sys
import subprocess
import importlib.util

print("Python executable:", sys.executable)
print("\nPackage status:")

subprocess.run([
    sys.executable, "-m", "pip", "show", "llama-cpp-python"
])

print("\nImportable module:", importlib.util.find_spec("llama_cpp"))

Python executable: /usr/bin/python3

Package status:

Importable module: ModuleSpec(name='llama_cpp', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7a5e44491850>, origin='/usr/local/lib/python3.13/dist-packages/llama_cpp/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/llama_cpp'])


In [36]:
%pip install --no-cache-dir --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu llama-cpp-python

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu


In [31]:
# Step 9G: Verify llama.cpp

import llama_cpp

print("Installation successful!")
print("Version:", llama_cpp.__version__)
print("GEO AI LLM runtime is ready!")

Installation successful!
Version: 0.3.36
GEO AI LLM runtime is ready!


In [33]:
# Step 10A: Download the quantized model

from huggingface_hub import hf_hub_download
import os

MODEL_REPO = "bartowski/SmolLM2-1.7B-Instruct-GGUF"
MODEL_FILE = "SmolLM2-1.7B-Instruct-Q4_K_M.gguf"

MODEL_PATH = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE
)

print("Model downloaded successfully!")
print("Model path:", MODEL_PATH)
print(f"Model file size: {os.path.getsize(MODEL_PATH) / (1024**3):.2f} GB")

Model downloaded successfully!
Model path: /root/.cache/huggingface/hub/models--bartowski--SmolLM2-1.7B-Instruct-GGUF/snapshots/1f03464768bfcc0319fc50da8ff5fb20b6417ba2/SmolLM2-1.7B-Instruct-Q4_K_M.gguf
Model file size: 0.98 GB


In [37]:
# Step 10B: Load the model

from llama_cpp import Llama

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_threads=2,
    n_threads_batch=2,
    n_gpu_layers=0,
    verbose=False
)

print("SmolLM2 loaded successfully!")

SmolLM2 loaded successfully!


In [38]:
# Step 10C: Test SmolLM2 with a GEO AI recommendation

test_prompt = """
You are GEO AI, an outdoor activity recommendation assistant.

Recommendation data:
Location: Bengaluru, Karnataka
Activity: Outdoor walking
Best time: Morning, 7:00 AM
Temperature: 21.2 degrees Celsius
PM2.5: 10.7 micrograms per cubic meter
US AQI: 43
Score: 100/100
Warnings: None generated by the scoring system

Explain in 3 short bullet points why this is a promising
time and location for outdoor walking.

Use only the supplied data. Do not invent weather conditions.
Mention that forecasts can change and that the score is
not a guarantee of safety.
"""

response = llm.create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": (
                "You are a careful environmental recommendation "
                "assistant. Be concise, factual, and transparent."
            )
        },
        {
            "role": "user",
            "content": test_prompt
        }
    ],
    temperature=0.2,
    max_tokens=180
)

print(response["choices"][0]["message"]["content"])

- Promising time and location for outdoor walking: 
  - Morning (7:00 AM) is a great time for walking as it is cooler and there is less traffic.
  - The temperature is pleasant at 21.2 degrees Celsius, making it comfortable for walking.
  - The PM2.5 level is 10.7 micrograms per cubic meter, which is relatively low and safe for outdoor activities.
  - The US AQI score of 43 indicates good air quality, reducing the risk of breathing issues.

However, it's essential to note that forecasts can change, and the score is not a guarantee of safety. Always check the current weather and air quality before heading out.


In [39]:

# Step 11D: Optimized LLM-powered GEO AI recommendations

def generate_llm_explanation(item):
    prompt = f"""
Explain this outdoor recommendation in exactly 3 short bullet points.
Use only the supplied data. Do not invent traffic, sunshine, or other conditions.
Do not claim the location or conditions are safe based only on the score.
Mention relevant warnings and that forecasts can change.

Location: {item['location']}, {item['state']}
Activity: {ACTIVITY}
Time: {item['forecast_time']} ({item['period']})
Temperature: {item.get('temperature_c')} C
PM2.5: {item.get('pm2_5')} µg/m³
US AQI: {item.get('us_aqi')}
Score: {item['score']}/100
Warnings: {item.get('warnings') or []}
Missing data: {item.get('missing_data') or []}
"""

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "You are GEO AI, a concise, evidence-based "
                    "environmental recommendation assistant. "
                    "Never invent data or overstate safety."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=100
    )

    return response["choices"][0]["message"]["content"].strip()


# Select the highest-scoring result for each time period
best_by_period = {}

for item in multi_time_results:
    period = item["period"]

    if (
        period not in best_by_period
        or item["score"] > best_by_period[period]["score"]
    ):
        best_by_period[period] = item


# Generate explanations for all three time periods
print("GEO AI — OPTIMIZED LLM RECOMMENDATIONS")
print("=" * 55)

for period in TIME_SLOTS:
    item = best_by_period.get(period)

    if item is None:
        print(f"\n{period}: No recommendation available.")
        continue

    print(f"\n{period}: {item['location']}, {item['state']}")
    print(f"Forecast: {item['forecast_time']}")
    print(f"Score: {item['score']}/100")
    print("-" * 45)

    try:
        explanation = generate_llm_explanation(item)
        print(explanation)
    except Exception as error:
        print("LLM generation failed:", repr(error))
        print("Please use the rule-based explanation as a fallback.")

print("\nStep 11D complete.")


GEO AI — OPTIMIZED LLM RECOMMENDATIONS

Morning: Bengaluru, Karnataka
Forecast: 2026-10-22T07:00
Score: 95/100
---------------------------------------------
- Location: Bengaluru, Karnataka
- Activity: Outdoor walking
- Time: 2026-10-22T07:00 (Morning)
- Temperature: 21.5 C
- PM2.5: 7.3 µg/m³
- US AQI: 53
- Score: 95/100
- Warnings: ['Check the precipitation forecast.']
- Missing data

Afternoon: Chandigarh, Chandigarh
Forecast: 2026-10-22T13:00
Score: 90/100
---------------------------------------------
- Location: Chandigarh, Chandigarh
- Activity: outdoor walking
- Time: 2026-10-22T13:00 (Afternoon)
- Temperature: 28.5 C
- PM2.5: 21.9 µg/m³
- US AQI: 102
- Score: 90/100
- Warnings: ['Consider air-quality conditions

Evening: Hyderabad, Telangana
Forecast: 2026-10-22T18:00
Score: 90/100
---------------------------------------------
- Location: Hyderabad, Telangana
- Activity: Outdoor walking
- Time: 2026-10-22T18:00 (Evening)
- Temperature: 29.5 C
- PM2.5: 34.5 µg/m³
- US AQI: 142
- 

In [40]:

# Step 12A: Generate concise, evidence-grounded explanations

def explain_recommendation(item):
    temperature = item.get("temperature_c")
    pm25 = item.get("pm2_5")
    aqi = item.get("us_aqi")
    warnings = item.get("warnings") or []
    missing = item.get("missing_data") or []

    prompt = f"""
You are GEO AI, explaining a recommendation for outdoor walking.

Data:
Location: {item['location']}, {item['state']}
Time: {item['period']} ({item['forecast_time']})
Temperature: {temperature} C
PM2.5: {pm25} micrograms/m3
US AQI: {aqi}
Rule-based score: {item['score']}/100
Warnings: {warnings}
Missing data: {missing}

Write exactly 3 short bullet points:
1. Explain what the temperature suggests for the activity, without
   claiming conditions not present in the data.
2. Explain the reported PM2.5 and AQI values without inventing
   an AQI category or claiming the air is safe.
3. Explain the score and any warnings.

End with one short sentence saying forecasts can change and the
score does not guarantee safety.

Do not repeat all the input fields. Do not invent facts.
"""

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "Be concise, factual, and cautious. "
                    "Never invent data or AQI classifications."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=130
    )

    return response["choices"][0]["message"]["content"].strip()


# Test one recommendation first to limit CPU workload
morning_item = best_by_period["Morning"]

print("GEO AI — IMPROVED EXPLANATION")
print("=" * 45)
print(
    f"{morning_item['location']}, {morning_item['state']} | "
    f"{morning_item['period']} | "
    f"Score: {morning_item['score']}/100"
)
print()
print(explain_recommendation(morning_item))


GEO AI — IMPROVED EXPLANATION
Bengaluru, Karnataka | Morning | Score: 95/100

1. The temperature of 21.5 C suggests a pleasant morning for outdoor walking, ideal for moderate activity.
2. The reported PM2.5 level of 7.3 micrograms/m3 indicates moderate air quality, which may require precautions if you have respiratory issues.
3. The score of 95/100 indicates a high recommendation for outdoor walking, but the AQI is 53, which is considered unhealthy. Warnings include checking the precipitation forecast.

Forecasts can change, and the score does not guarantee safety.


In [41]:

# Step 12B: Verify facts before sending them to the LLM

def get_us_aqi_category(aqi):
    if aqi is None:
        return "unavailable"
    if aqi <= 50:
        return "Good"
    if aqi <= 100:
        return "Moderate"
    if aqi <= 150:
        return "Unhealthy for Sensitive Groups"
    if aqi <= 200:
        return "Unhealthy"
    if aqi <= 300:
        return "Very Unhealthy"
    return "Hazardous"


def explain_verified_recommendation(item):
    temperature = item.get("temperature_c")
    pm25 = item.get("pm2_5")
    aqi = item.get("us_aqi")
    score = item["score"]
    warnings = item.get("warnings") or []

    # These factual classifications are determined by Python,
    # not generated by the language model.
    aqi_category = get_us_aqi_category(aqi)

    if temperature is None:
        temperature_fact = "Temperature forecast unavailable."
    else:
        temperature_fact = (
            f"Forecast temperature is {temperature} degrees Celsius."
        )

    if pm25 is None:
        pm25_fact = "PM2.5 forecast unavailable."
    else:
        pm25_fact = (
            f"Forecast PM2.5 is {pm25} micrograms per cubic meter."
        )

    if aqi is None:
        aqi_fact = "US AQI forecast unavailable."
    else:
        aqi_fact = (
            f"Forecast US AQI is {aqi}; its US AQI category is "
            f"{aqi_category}."
        )

    facts = f"""
Location: {item['location']}, {item['state']}
Activity: {ACTIVITY}
Time: {item['period']} ({item['forecast_time']})
{temperature_fact}
{pm25_fact}
{aqi_fact}
Application's rule-based score: {score}/100.
Scoring-engine warnings: {warnings}
"""

    prompt = f"""
Explain these verified GEO AI facts in exactly 3 concise bullet points.

{facts}

Explain the temperature, pollution measurements, and application score.
Do not change any numbers or AQI category.
Do not call the score an AQI score.
Do not claim the location is safe.
Do not invent missing information.
End with: "Forecasts can change; this score is not a guarantee of safety."
"""

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "Explain supplied facts accurately. "
                    "Never change numeric values or classifications."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=150
    )

    return response["choices"][0]["message"]["content"].strip()


# Test Bengaluru's best morning recommendation
morning_item = best_by_period["Morning"]

print("VERIFIED GEO AI RECOMMENDATION")
print("=" * 45)
print("Location:", morning_item["location"])
print("Rule-based score:", morning_item["score"], "/100")
print("US AQI category:", get_us_aqi_category(morning_item["us_aqi"]))
print()
print(explain_verified_recommendation(morning_item))


VERIFIED GEO AI RECOMMENDATION
Location: Bengaluru
Rule-based score: 95 /100
US AQI category: Moderate

Location: Bengaluru, Karnataka
Activity: outdoor walking
Time: Morning (2026-10-22T07:00)
Forecast temperature: 21.5 degrees Celsius
Forecast PM2.5: 7.3 micrograms per cubic meter
Forecast US AQI: 53; US AQI category: Moderate
Application's rule-based score: 95/100
Scoring-engine warnings: ['Check the precipitation forecast.']
Forecasts can change; this score is not a guarantee of safety.


In [42]:

# Step 12C: Produce concise, verified GEO AI explanations

def final_geo_explanation(item):
    temp = item.get("temperature_c")
    pm25 = item.get("pm2_5")
    aqi = item.get("us_aqi")
    score = item["score"]
    warnings = item.get("warnings") or []

    category = get_us_aqi_category(aqi)

    # Python creates the verified facts.
    facts = [
        f"Location: {item['location']}, {item['state']}.",
        f"Activity: {ACTIVITY}.",
        f"Recommended period: {item['period']} at {item['forecast_time']}.",
        (
            f"Temperature: {temp} °C."
            if temp is not None else "Temperature: unavailable."
        ),
        (
            f"PM2.5: {pm25} µg/m³."
            if pm25 is not None else "PM2.5: unavailable."
        ),
        (
            f"US AQI: {aqi}, category {category}."
            if aqi is not None else "US AQI: unavailable."
        ),
        f"Application score: {score}/100.",
        f"Scoring warnings: {warnings or 'None generated'}.",
    ]

    prompt = f"""
Write exactly 3 short bullet points for a GEO AI recommendation.

Use these verified facts:
{chr(10).join(facts)}

Rules:
- Explain the data; do not merely list every field again.
- Do not invent weather, traffic, or health outcomes.
- Preserve the AQI category and all numeric values.
- Do not call the application score an AQI score.
- Mention warnings when present.
- Do not claim the conditions are safe.
- Finish with: Forecasts can change; this score is not a guarantee of safety.
- Keep the entire answer under 100 words.
"""

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "You are GEO AI. Explain environmental forecast "
                    "data cautiously. Never invent facts."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=130
    )

    explanation = response["choices"][0]["message"]["content"].strip()

    print("VERIFIED INPUT FACTS")
    for fact in facts:
        print("•", fact)

    print("\nLLM EXPLANATION")
    print(explanation)


# Test with the best morning recommendation
final_geo_explanation(best_by_period["Morning"])


VERIFIED INPUT FACTS
• Location: Bengaluru, Karnataka.
• Activity: outdoor walking.
• Recommended period: Morning at 2026-10-22T07:00.
• Temperature: 21.5 °C.
• PM2.5: 7.3 µg/m³.
• US AQI: 53, category Moderate.
• Application score: 95/100.
• Scoring warnings: ['Check the precipitation forecast.'].

LLM EXPLANATION
- The recommended period for outdoor walking in Bengaluru on 2026-10-22T07:00 is the morning, with a temperature of 21.5 °C.
- The PM2.5 level is 7.3 µg/m³, which falls within the Moderate category of the US AQI.
- The application score is 95/100, indicating a high level of accuracy in the forecast.

Forecasts can change; this score is not a guarantee of safety.


In [43]:

# Step 12D: Reliable, fact-checked GEO AI explanation

def reliable_geo_explanation(item):
    location = f"{item['location']}, {item['state']}"
    period = item["period"]
    forecast_time = item["forecast_time"]
    temperature = item.get("temperature_c")
    pm25 = item.get("pm2_5")
    aqi = item.get("us_aqi")
    score = item["score"]
    warnings = item.get("warnings") or []
    missing = item.get("missing_data") or []

    print(f"GEO AI RECOMMENDATION: {location}")
    print("=" * 55)
    print(f"Activity: {ACTIVITY}")
    print(f"Recommended period: {period} ({forecast_time})")
    print(f"Application score: {score}/100")

    print("\nWHY THIS OPTION RANKED THIS WAY")

    if temperature is None:
        print("- Temperature forecast is unavailable.")
    elif temperature <= 25:
        print(
            f"- The forecast temperature is {temperature} °C, "
            "which is relatively mild for outdoor walking."
        )
    elif temperature >= 35:
        print(
            f"- The forecast temperature is {temperature} °C; "
            "heat exposure deserves consideration."
        )
    else:
        print(
            f"- The forecast temperature is {temperature} °C. "
            "Consider personal comfort and conditions."
        )

    if pm25 is None:
        print("- The PM2.5 forecast is unavailable.")
    else:
        print(
            f"- Forecast PM2.5 is {pm25} µg/m³. "
            "Consider this measurement alongside the AQI and "
            "any personal health considerations."
        )

    if aqi is None:
        print("- The US AQI forecast is unavailable.")
    else:
        category = get_us_aqi_category(aqi)
        print(f"- Forecast US AQI is {aqi} ({category}).")

    print(f"- The rule-based application score is {score}/100.")

    if warnings:
        print("\nWARNINGS")
        for warning in warnings:
            print(f"- {warning}")
    else:
        print("\nWarnings: None generated by the scoring engine.")

    if missing:
        print("\nMissing data:", missing)

    print(
        "\nNote: Forecasts can change. This score is not a "
        "guarantee of safety."
    )


# Test the highest-ranked morning recommendation
reliable_geo_explanation(best_by_period["Morning"])


GEO AI RECOMMENDATION: Bengaluru, Karnataka
Activity: outdoor walking
Recommended period: Morning (2026-10-22T07:00)
Application score: 95/100

WHY THIS OPTION RANKED THIS WAY
- The forecast temperature is 21.5 °C, which is relatively mild for outdoor walking.
- Forecast PM2.5 is 7.3 µg/m³. Consider this measurement alongside the AQI and any personal health considerations.
- Forecast US AQI is 53 (Moderate).
- The rule-based application score is 95/100.

WARNINGS
- Check the precipitation forecast.

Note: Forecasts can change. This score is not a guarantee of safety.


In [44]:

# STEP 13A: GEO AI User Personalization
# Create a NEW cell. Do not replace any existing cells.

print("🌍 GEO AI — PERSONALIZED OUTDOOR PLANNER")
print("=" * 48)

# 1. Choose an activity
available_activities = [
    "outdoor walking",
    "running",
    "cycling",
    "hiking",
    "picnic",
    "outdoor exercise"
]

print("\nChoose your activity:")
for i, activity in enumerate(available_activities, start=1):
    print(f"{i}. {activity}")

while True:
    choice = input("Enter the activity number (1-6): ").strip()
    if choice in [str(i) for i in range(1, len(available_activities) + 1)]:
        USER_ACTIVITY = available_activities[int(choice) - 1]
        break
    print("Invalid choice. Please enter a number from 1 to 6.")

# 2. Health considerations
print("\nEnter any health considerations separated by commas.")
print("Examples: asthma, heart condition, none")

health_input = input(
    "Health considerations (press Enter for none): "
).strip()

if not health_input or health_input.lower() == "none":
    USER_HEALTH_CONDITIONS = []
else:
    USER_HEALTH_CONDITIONS = [
        condition.strip().lower()
        for condition in health_input.split(",")
        if condition.strip()
    ]

# 3. Forecast date
# Open-Meteo forecast availability is limited, so validate the date
# against the forecast data later rather than assuming every date works.

from datetime import datetime
default_date = datetime.now().strftime("%Y-%m-%d")

date_input = input(
    f"\nForecast date (YYYY-MM-DD; Enter for {default_date}): "
).strip()

USER_TARGET_DATE = date_input if date_input else default_date

# 4. Show the selected preferences
print("\n" + "=" * 48)
print("YOUR GEO AI PREFERENCES")
print("Activity:", USER_ACTIVITY)
print(
    "Health considerations:",
    USER_HEALTH_CONDITIONS if USER_HEALTH_CONDITIONS else "None provided"
)
print("Forecast date:", USER_TARGET_DATE)
print("=" * 48)


🌍 GEO AI — PERSONALIZED OUTDOOR PLANNER

Choose your activity:
1. outdoor walking
2. running
3. cycling
4. hiking
5. picnic
6. outdoor exercise
Enter the activity number (1-6): 1

Enter any health considerations separated by commas.
Examples: asthma, heart condition, none
Health considerations (press Enter for none): asthma

Forecast date (YYYY-MM-DD; Enter for 2026-10-22): 2026-10-23

YOUR GEO AI PREFERENCES
Activity: outdoor walking
Health considerations: ['asthma']
Forecast date: 2026-10-23


In [46]:

# STEP 13B-FIX: Restore the city coordinate lookup helper
import requests

def get_coordinates(city):
    """Return (latitude, longitude) for a city using Open-Meteo geocoding."""
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": city,
        "count": 10,
        "language": "en",
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    matches = data.get("results", [])
    if not matches:
        raise ValueError(f"No coordinates found for {city}")

    # Prefer a match in India to avoid similarly named cities elsewhere.
    india_matches = [
        place for place in matches
        if place.get("country_code") == "IN"
    ]
    place = india_matches[0] if india_matches else matches[0]

    return place["latitude"], place["longitude"]


# Test the helper before rerunning all comparisons.
print("Pune coordinates:", get_coordinates("Pune"))
print("Hyderabad coordinates:", get_coordinates("Hyderabad"))


Pune coordinates: (18.51957, 73.85535)
Hyderabad coordinates: (17.38405, 78.45636)


In [56]:

# GEO AI — SIMPLE PERSONALIZED OUTDOOR PLANNER 🌍
import requests

BASE = "https://api.open-meteo.com/v1/forecast"
AIR_BASE = "https://air-quality-api.open-meteo.com/v1/air-quality"
GEO_BASE = "https://geocoding-api.open-meteo.com/v1/search"
TZ = "Asia/Kolkata"

# Reuse Step 13A preferences if available; otherwise ask for them.
activity = globals().get("USER_ACTIVITY") or input(
    "Activity (e.g. outdoor walking): "
).strip()

health = globals().get("USER_HEALTH_CONDITIONS", [])
if not health:
    health_input = input("Health considerations (Enter for none): ").strip()
    health = [x.strip().lower() for x in health_input.split(",") if x.strip()]
else:
    health = [str(x).lower() for x in health]

target_date = globals().get("USER_TARGET_DATE") or input(
    "Forecast date (YYYY-MM-DD): "
).strip()

cities = [
    "Pune", "Mumbai", "Hyderabad", "Bengaluru", "Kolkata",
    "Jaipur", "Ahmedabad", "Chandigarh", "Delhi"
]

time_slots = [
    ("Morning", "07:00"),
    ("Afternoon", "13:00"),
    ("Evening", "18:00")
]

def get_place(city):
    r = requests.get(GEO_BASE, params={
        "name": city, "count": 10, "language": "en", "format": "json"
    }, timeout=20)
    r.raise_for_status()
    places = r.json().get("results", [])
    places = [p for p in places if p.get("country_code") == "IN"]
    if not places:
        raise ValueError("Indian location not found")
    p = places[0]
    return p["latitude"], p["longitude"], p.get("admin1", "India")

def get_value(data, field, timestamp):
    hourly = data.get("hourly", {})
    times = hourly.get("time", [])
    for i, t in enumerate(times):
        if t == timestamp:
            values = hourly.get(field, [])
            if i < len(values):
                return values[i]
    return None

def score_option(temp, rain, pm25, aqi):
    score = 100
    warnings = []

    if temp is not None:
        if temp >= 36:
            score -= 25
            warnings.append("Very hot conditions")
        elif temp >= 30:
            score -= 10
            warnings.append("Warm conditions")

    if rain is not None and rain >= 50:
        score -= 15
        warnings.append("High chance of rain")

    if pm25 is not None:
        if pm25 >= 55:
            score -= 25
            warnings.append("High PM2.5")
        elif pm25 >= 35:
            score -= 15
            warnings.append("Elevated PM2.5")
        elif pm25 >= 15:
            score -= 5
            warnings.append("Consider PM2.5 levels")

    if aqi is not None:
        if aqi >= 151:
            score -= 25
            warnings.append("Unhealthy AQI")
        elif aqi >= 101:
            score -= 15
            warnings.append("Elevated AQI")
        elif aqi >= 51:
            score -= 5
            warnings.append("AQI above Good")

    if "asthma" in health and (
        (pm25 is not None and pm25 >= 15) or
        (aqi is not None and aqi >= 51)
    ):
        warnings.append("Asthma: review air quality before going out")

    return max(0, min(100, score)), warnings

results = []
unavailable = []

print("Fetching forecasts. Please wait...\n")

for city in cities:
    try:
        lat, lon, state = get_place(city)

        weather_response = requests.get(BASE, params={
            "latitude": lat,
            "longitude": lon,
            "hourly": "temperature_2m,precipitation_probability",
            "start_date": target_date,
            "end_date": target_date,
            "timezone": TZ
        }, timeout=30)
        weather_response.raise_for_status()
        weather = weather_response.json()

        air_response = requests.get(AIR_BASE, params={
            "latitude": lat,
            "longitude": lon,
            "hourly": "pm2_5,us_aqi",
            "start_date": target_date,
            "end_date": target_date,
            "timezone": TZ
        }, timeout=30)
        air_response.raise_for_status()
        air = air_response.json()

        for period, hour in time_slots:
            timestamp = f"{target_date}T{hour}"

            temp = get_value(weather, "temperature_2m", timestamp)
            rain = get_value(weather, "precipitation_probability", timestamp)
            pm25 = get_value(air, "pm2_5", timestamp)
            aqi = get_value(air, "us_aqi", timestamp)

            if temp is None or (pm25 is None and aqi is None):
                unavailable.append(f"{city} — {period}: forecast data missing")
                continue

            score, warnings = score_option(temp, rain, pm25, aqi)

            results.append({
                "city": city,
                "state": state,
                "period": period,
                "time": timestamp,
                "temp": temp,
                "rain": rain,
                "pm25": pm25,
                "aqi": aqi,
                "score": score,
                "warnings": warnings
            })

    except Exception as e:
        unavailable.append(f"{city}: {str(e)[:120]}")

results.sort(key=lambda x: x["score"], reverse=True)

print("=" * 48)
print("GEO AI — PERSONALIZED RECOMMENDATIONS")
print("=" * 48)
print("Activity:", activity)
print("Health considerations:", ", ".join(health) if health else "None entered")
print("Forecast date:", target_date)
print("Options evaluated:", len(results))

if results:
    print("\nTOP 5 OPTIONS\n")

    for rank, r in enumerate(results[:5], 1):
        print(f"#{rank} {r['city']}, {r['state']} — {r['period']}")
        print(f"   Time: {r['time']}")
        print(f"   Score: {r['score']}/100")
        print(f"   Temperature: {r['temp']} °C")
        print(f"   PM2.5: {r['pm25']} µg/m³")
        print(f"   US AQI: {r['aqi']}")
        print(f"   Rain chance: {r['rain']}%")
        print("   Warnings:", "; ".join(r["warnings"]) if r["warnings"] else "None from these rules")
        print()

    best = results[0]
    print("BEST MATCH:", best["city"], "-", best["period"])
    print("This is the highest-scoring available option, not a guarantee of safety.")
    print("Check updated forecasts and personal health needs before going out.")
else:
    print("No usable results. Check the forecast date and internet access.")

if unavailable:
    print("\nCITIES / TIMES NOT CHECKED")
    for item in unavailable:
        print("-", item)


Fetching forecasts. Please wait...

GEO AI — PERSONALIZED RECOMMENDATIONS
Activity: outdoor walking
Health considerations: asthma
Forecast date: 2026-10-23
Options evaluated: 27

TOP 5 OPTIONS

#1 Bengaluru, Karnataka — Morning
   Time: 2026-10-23T07:00
   Score: 100/100
   Temperature: 21.4 °C
   PM2.5: 4.6 µg/m³
   US AQI: 38
   Rain chance: 9%
   Warnings: None from these rules

#2 Bengaluru, Karnataka — Afternoon
   Time: 2026-10-23T13:00
   Score: 100/100
   Temperature: 27.2 °C
   PM2.5: 4.5 µg/m³
   US AQI: 37
   Rain chance: 44%
   Warnings: None from these rules

#3 Chandigarh, Chandigarh — Afternoon
   Time: 2026-10-23T13:00
   Score: 90/100
   Temperature: 29.0 °C
   PM2.5: 22.9 µg/m³
   US AQI: 100
   Rain chance: 0%
   Warnings: Consider PM2.5 levels; AQI above Good; Asthma: review air quality before going out

#4 Pune, Maharashtra — Morning
   Time: 2026-10-23T07:00
   Score: 80/100
   Temperature: 23.5 °C
   PM2.5: 39.5 µg/m³
   US AQI: 97
   Rain chance: 0%
   Warnings:

In [57]:

# ============================================================
# GEO AI — STEP 13B: PERSONALIZED COMPARISON ENGINE
# Creates personalized_results without overwriting previous data.
# Requires the existing score_conditions() function and Step 13A
# preference variables: USER_ACTIVITY, USER_HEALTH_CONDITIONS,
# USER_TARGET_DATE.
# ============================================================

import requests
from datetime import datetime
from zoneinfo import ZoneInfo

PERSONALIZED_CITIES = [
    "Pune", "Mumbai", "Hyderabad", "Bengaluru", "Kolkata",
    "Jaipur", "Ahmedabad", "Chandigarh", "Delhi"
]

PERSONALIZED_TIMEZONE = "Asia/Kolkata"

PERSONALIZED_TIME_SLOTS = {
    "Morning": "07:00",
    "Afternoon": "13:00",
    "Evening": "18:00"
}

# Check required inputs and scoring function before making API calls.
required_names = [
    "USER_ACTIVITY",
    "USER_HEALTH_CONDITIONS",
    "USER_TARGET_DATE",
    "score_conditions"
]

missing_names = [
    name for name in required_names
    if name not in globals()
]

if missing_names:
    raise NameError(
        "Missing required definitions: "
        + ", ".join(missing_names)
        + ". Run Step 13A and the cell defining score_conditions() first."
    )

# Validate the date format.
try:
    datetime.strptime(USER_TARGET_DATE, "%Y-%m-%d")
except ValueError:
    raise ValueError("USER_TARGET_DATE must use YYYY-MM-DD format.")

session = requests.Session()

def personalized_get_coordinates(city):
    """Find a city's coordinates, preferring matches in India."""
    response = session.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={
            "name": city,
            "count": 10,
            "language": "en",
            "format": "json"
        },
        timeout=30
    )
    response.raise_for_status()

    matches = response.json().get("results", [])
    if not matches:
        raise ValueError(f"No coordinates found for {city}")

    india_matches = [
        place for place in matches
        if place.get("country_code") == "IN"
    ]

    place = india_matches[0] if india_matches else matches[0]

    return {
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "city": place.get("name", city),
        "state": place.get("admin1", "India")
    }


def personalized_hour_index(times, target_hour, city, data_type):
    """Match an hour against API timestamps such as YYYY-MM-DDTHH:MM."""
    target_timestamp = f"{USER_TARGET_DATE}T{target_hour}"

    for index, timestamp in enumerate(times):
        if timestamp == target_timestamp:
            return index

    # Include available timestamps in the error for diagnosis.
    raise ValueError(
        f"{data_type} forecast unavailable for {city} at "
        f"{target_timestamp}. Sample available timestamps: {times[:3]}"
    )


def personalized_hourly_value(hourly, key, index):
    values = hourly.get(key, [])
    if index >= len(values):
        return None
    return values[index]


def compare_city_personalized(city, period, target_hour):
    """Fetch forecasts and score them using the user's preferences."""

    coordinates = personalized_get_coordinates(city)
    lat = coordinates["latitude"]
    lon = coordinates["longitude"]

    # Weather forecast for the selected date.
    weather_response = session.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "hourly": (
                "temperature_2m,apparent_temperature,"
                "precipitation_probability,precipitation,"
                "relative_humidity_2m,wind_speed_10m,weather_code"
            ),
            "timezone": PERSONALIZED_TIMEZONE,
            "start_date": USER_TARGET_DATE,
            "end_date": USER_TARGET_DATE
        },
        timeout=30
    )
    weather_response.raise_for_status()
    weather_payload = weather_response.json()

    weather_hourly = weather_payload.get("hourly", {})
    weather_times = weather_hourly.get("time", [])

    weather_idx = personalized_hour_index(
        weather_times, target_hour, city, "Weather"
    )

    weather = {
        "temperature": personalized_hourly_value(
            weather_hourly, "temperature_2m", weather_idx
        ),
        "feels_like": personalized_hourly_value(
            weather_hourly, "apparent_temperature", weather_idx
        ),
        "precipitation_probability": personalized_hourly_value(
            weather_hourly, "precipitation_probability", weather_idx
        ),
        "precipitation": personalized_hourly_value(
            weather_hourly, "precipitation", weather_idx
        ),
        "humidity": personalized_hourly_value(
            weather_hourly, "relative_humidity_2m", weather_idx
        ),
        "wind_speed": personalized_hourly_value(
            weather_hourly, "wind_speed_10m", weather_idx
        ),
        "weather_code": personalized_hourly_value(
            weather_hourly, "weather_code", weather_idx
        )
    }

    # Air-quality forecast for the same location, date, and hour.
    air_response = session.get(
        "https://air-quality-api.open-meteo.com/v1/air-quality",
        params={
            "latitude": lat,
            "longitude": lon,
            "hourly": (
                "pm2_5,pm10,us_aqi,european_aqi,"
                "ozone,nitrogen_dioxide"
            ),
            "timezone": PERSONALIZED_TIMEZONE,
            "start_date": USER_TARGET_DATE,
            "end_date": USER_TARGET_DATE
        },
        timeout=30
    )
    air_response.raise_for_status()
    air_payload = air_response.json()

    air_hourly = air_payload.get("hourly", {})
    air_times = air_hourly.get("time", [])

    air_idx = personalized_hour_index(
        air_times, target_hour, city, "Air-quality"
    )

    air_quality = {
        "pm2_5": personalized_hourly_value(
            air_hourly, "pm2_5", air_idx
        ),
        "pm10": personalized_hourly_value(
            air_hourly, "pm10", air_idx
        ),
        "us_aqi": personalized_hourly_value(
            air_hourly, "us_aqi", air_idx
        ),
        "european_aqi": personalized_hourly_value(
            air_hourly, "european_aqi", air_idx
        ),
        "ozone": personalized_hourly_value(
            air_hourly, "ozone", air_idx
        ),
        "nitrogen_dioxide": personalized_hourly_value(
            air_hourly, "nitrogen_dioxide", air_idx
        )
    }

    # Reuse the project's existing scoring logic.
    score, warnings = score_conditions(
        USER_ACTIVITY,
        weather,
        air_quality,
        USER_HEALTH_CONDITIONS
    )

    # Keep location metadata and data separate from old results.
    missing_data = [
        key for key, value in {
            **weather, **air_quality
        }.items()
        if value is None
    ]

    return {
        "location": coordinates["city"],
        "state": coordinates["state"],
        "latitude": lat,
        "longitude": lon,
        "period": period,
        "forecast_time": f"{USER_TARGET_DATE}T{target_hour}",
        "temperature": weather["temperature"],
        "temperature_c": weather["temperature"],
        "pm2_5": air_quality["pm2_5"],
        "us_aqi": air_quality["us_aqi"],
        "score": score,
        "warnings": warnings or [],
        "missing_data": missing_data,
        "activity": USER_ACTIVITY,
        "health_conditions": USER_HEALTH_CONDITIONS
    }



# Run all 9 cities x 3 periods.
personalized_results = []
personalized_unavailable = []

print("Starting personalized GEO AI comparisons...")
print("Date:", USER_TARGET_DATE)
print("Activity:", USER_ACTIVITY)
print("Health considerations:", USER_HEALTH_CONDITIONS or "None")
print()

for city in PERSONALIZED_CITIES:
    for period, target_hour in PERSONALIZED_TIME_SLOTS.items():
        try:
            item = compare_city_personalized(
                city, period, target_hour
            )
            personalized_results.append(item)
            print(
                f"✓ {city} — {period}: "
                f"{item['score']}/100"
            )
        except Exception as error:
            personalized_unavailable.append({
                "city": city,
                "period": period,
                "error": str(error)
            })
            print(f"✗ {city} — {period}: {error}")

personalized_results.sort(
    key=lambda item: item["score"],
    reverse=True
)

print("\n" + "=" * 58)
print("GEO AI — PERSONALIZED COMPARISON SUMMARY")
print("=" * 58)
print("Activity:", USER_ACTIVITY)
print("Health considerations:", USER_HEALTH_CONDITIONS or "None")
print("Forecast date:", USER_TARGET_DATE)
print(
    f"Successful comparisons: {len(personalized_results)}"
    f" / {len(PERSONALIZED_CITIES) * len(PERSONALIZED_TIME_SLOTS)}"
)
print("Unavailable comparisons:", len(personalized_unavailable))

print("\nTOP 5 PERSONALIZED RESULTS")

for rank, item in enumerate(personalized_results[:5], start=1):
    print(
        f"\n#{rank} {item['location']}, {item['state']} "
        f"— {item['period']}"
    )
    print("Forecast:", item["forecast_time"])
    print("Activity:", item["activity"])
    print("Temperature:", item["temperature_c"], "°C")
    print("PM2.5:", item["pm2_5"], "µg/m³")
    print("US AQI:", item["us_aqi"])
    print("Application score:", f"{item['score']}/100")
    print("Warnings:", item["warnings"] or "None generated")
    if item["missing_data"]:
        print("Missing data:", item["missing_data"])

if personalized_unavailable:
    print("\nUNAVAILABLE COMPARISONS")
    for item in personalized_unavailable:
        print(
            f"- {item['city']} / {item['period']}: "
            f"{item['error']}"
        )

if not personalized_results:
    print(
        "\nNo recommendations could be generated. "
        "Check the errors above before relying on results."
    )
elif len(personalized_results) < 27:
    print(
        "\nSome comparisons failed. Use only the successful results "
        "and investigate unavailable entries before presenting the demo."
    )
else:
    print(
        "\nAll 27 comparisons completed. Forecasts can change; "
        "scores are not guarantees of safety."
    )


Starting personalized GEO AI comparisons...
Date: 2026-10-23
Activity: outdoor walking
Health considerations: ['asthma']

✗ Pune — Morning: too many values to unpack (expected 2)
✗ Pune — Afternoon: too many values to unpack (expected 2)
✗ Pune — Evening: too many values to unpack (expected 2)
✗ Mumbai — Morning: too many values to unpack (expected 2)
✗ Mumbai — Afternoon: too many values to unpack (expected 2)
✗ Mumbai — Evening: too many values to unpack (expected 2)
✗ Hyderabad — Morning: too many values to unpack (expected 2)
✗ Hyderabad — Afternoon: too many values to unpack (expected 2)
✗ Hyderabad — Evening: too many values to unpack (expected 2)
✗ Bengaluru — Morning: too many values to unpack (expected 2)
✗ Bengaluru — Afternoon: too many values to unpack (expected 2)
✗ Bengaluru — Evening: too many values to unpack (expected 2)
✗ Kolkata — Morning: too many values to unpack (expected 2)
✗ Kolkata — Afternoon: too many values to unpack (expected 2)
✗ Kolkata — Evening: too man

In [1]:
import inspect

for name in ["get_hourly_data", "compare_city", "score_conditions"]:
    fn = globals().get(name)
    print(f"\n--- {name} ---")
    if callable(fn):
        try:
            print(inspect.getsource(fn))
        except Exception:
            print("Source unavailable")
    else:
        print("Not defined")


--- get_hourly_data ---
Not defined

--- compare_city ---
Not defined

--- score_conditions ---
Not defined


In [3]:

# GEO AI — CLEAN PERSONALIZED COMPARISON 🌍
import requests

TZ = "Asia/Kolkata"
GEO_URL = "https://geocoding-api.open-meteo.com/v1/search"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
AIR_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

activity = globals().get("USER_ACTIVITY") or input(
    "Activity (e.g. outdoor walking): "
).strip()

health = globals().get("USER_HEALTH_CONDITIONS", [])
if isinstance(health, str):
    health = [x.strip().lower() for x in health.split(",") if x.strip()]
else:
    health = [str(x).lower() for x in health]

if not health:
    answer = input("Health considerations (Enter for none): ").strip()
    health = [x.strip().lower() for x in answer.split(",") if x.strip()]

date = globals().get("USER_TARGET_DATE") or input(
    "Forecast date (YYYY-MM-DD): "
).strip()

cities = [
    "Pune", "Mumbai", "Hyderabad", "Bengaluru", "Kolkata",
    "Jaipur", "Ahmedabad", "Chandigarh", "Delhi"
]
slots = [("Morning", "07:00"), ("Afternoon", "13:00"), ("Evening", "18:00")]

def geo_get(url, params):
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

def find_city(city):
    data = geo_get(GEO_URL, {
        "name": city, "count": 10, "language": "en", "format": "json"
    })
    places = [
        p for p in data.get("results", [])
        if p.get("country_code") == "IN"
    ]
    if not places:
        raise ValueError("Indian city not found")
    p = places[0]
    return p["latitude"], p["longitude"], p.get("admin1", "India")

def hourly_value(data, field, timestamp):
    hourly = data.get("hourly", {})
    times = hourly.get("time", [])
    values = hourly.get(field, [])
    for i, t in enumerate(times):
        if t == timestamp and i < len(values):
            return values[i]
    return None

def rank_score(temp, rain, pm25, aqi, health):
    score, warnings = 100, []

    if temp is not None:
        if temp >= 36:
            score -= 25
            warnings.append("Very hot")
        elif temp >= 30:
            score -= 10
            warnings.append("Hot")

    if rain is not None and rain >= 50:
        score -= 15
        warnings.append("High rain probability")

    if pm25 is not None:
        if pm25 >= 55:
            score -= 25
            warnings.append("High PM2.5")
        elif pm25 >= 35:
            score -= 15
            warnings.append("Elevated PM2.5")
        elif pm25 >= 15:
            score -= 5
            warnings.append("Consider PM2.5")

    if aqi is not None:
        if aqi >= 151:
            score -= 25
            warnings.append("Unhealthy AQI")
        elif aqi >= 101:
            score -= 15
            warnings.append("Elevated AQI")
        elif aqi >= 51:
            score -= 5
            warnings.append("AQI above Good")

    if "asthma" in health and (
        (pm25 is not None and pm25 >= 15) or
        (aqi is not None and aqi >= 51)
    ):
        warnings.append("Asthma: review air quality")

    return max(0, score), warnings

results, unavailable = [], []

print("\nStarting personalized GEO AI comparisons...")
print("Date:", date)
print("Activity:", activity)
print("Health considerations:", health or "None")

for city in cities:
    try:
        lat, lon, state = find_city(city)

        weather = geo_get(WEATHER_URL, {
            "latitude": lat,
            "longitude": lon,
            "hourly": "temperature_2m,precipitation_probability",
            "start_date": date,
            "end_date": date,
            "timezone": TZ
        })

        air = geo_get(AIR_URL, {
            "latitude": lat,
            "longitude": lon,
            "hourly": "pm2_5,us_aqi",
            "start_date": date,
            "end_date": date,
            "timezone": TZ
        })

        for period, hour in slots:
            timestamp = f"{date}T{hour}"
            temp = hourly_value(weather, "temperature_2m", timestamp)
            rain = hourly_value(weather, "precipitation_probability", timestamp)
            pm25 = hourly_value(air, "pm2_5", timestamp)
            aqi = hourly_value(air, "us_aqi", timestamp)

            if temp is None or (pm25 is None and aqi is None):
                unavailable.append(f"{city} / {period}: required forecast data missing")
                continue

            score, warnings = rank_score(temp, rain, pm25, aqi, health)

            results.append({
                "city": city, "state": state, "period": period,
                "time": timestamp, "temp": temp, "rain": rain,
                "pm25": pm25, "aqi": aqi, "score": score,
                "warnings": warnings
            })

    except Exception as e:
        unavailable.append(f"{city}: {type(e).__name__}: {e}")

results.sort(key=lambda x: x["score"], reverse=True)

print(f"\nSuccessful comparisons: {len(results)} / 27")
print(f"Unavailable comparisons: {len(unavailable)}")

if results:
    print("\nTOP 5 PERSONALIZED RESULTS\n")
    for n, r in enumerate(results[:5], 1):
        print(f"#{n} {r['city']}, {r['state']} — {r['period']}")
        print(f"   Time: {r['time']}")
        print(f"   Score: {r['score']}/100")
        print(f"   Temperature: {r['temp']} °C")
        print(f"   PM2.5: {r['pm25']} µg/m³")
        print(f"   US AQI: {r['aqi']}")
        print(f"   Rain chance: {r['rain']}%")
        print("   Warnings:", "; ".join(r["warnings"]) or "None from these rules")
        print()
else:
    print("\nNo recommendations could be generated.")

if unavailable:
    print("\nUNAVAILABLE COMPARISONS")
    for item in unavailable:
        print("-", item)

print("\nScores are prototype rankings, not guarantees of safety.")


Activity (e.g. outdoor walking): outdoor walking
Health considerations (Enter for none): asthma
Forecast date (YYYY-MM-DD): 2026-10-25

Starting personalized GEO AI comparisons...
Date: 2026-10-25
Activity: outdoor walking
Health considerations: ['asthma']

Successful comparisons: 27 / 27
Unavailable comparisons: 0

TOP 5 PERSONALIZED RESULTS

#1 Hyderabad, Telangana — Morning
   Time: 2026-10-25T07:00
   Score: 90/100
   Temperature: 25.0 °C
   PM2.5: 28.9 µg/m³
   US AQI: 100
   Rain chance: 0%
   Warnings: Consider PM2.5; AQI above Good; Asthma: review air quality

#2 Jaipur, Rajasthan — Morning
   Time: 2026-10-25T07:00
   Score: 90/100
   Temperature: 22.4 °C
   PM2.5: 32.2 µg/m³
   US AQI: 79
   Rain chance: 0%
   Warnings: Consider PM2.5; AQI above Good; Asthma: review air quality

#3 Jaipur, Rajasthan — Evening
   Time: 2026-10-25T18:00
   Score: 90/100
   Temperature: 27.8 °C
   PM2.5: 29.1 µg/m³
   US AQI: 79
   Rain chance: 0%
   Warnings: Consider PM2.5; AQI above Good; Ast

In [48]:

# ============================================================
# GEO AI — STEP 13B: PERSONALIZED COMPARISON ENGINE
# Creates personalized_results without overwriting previous data.
# Requires the existing score_conditions() function and Step 13A
# preference variables: USER_ACTIVITY, USER_HEALTH_CONDITIONS,
# USER_TARGET_DATE.
# ============================================================

import requests
from datetime import datetime
from zoneinfo import ZoneInfo

PERSONALIZED_CITIES = [
    "Pune", "Mumbai", "Hyderabad", "Bengaluru", "Kolkata",
    "Jaipur", "Ahmedabad", "Chandigarh", "Delhi"
]

PERSONALIZED_TIMEZONE = "Asia/Kolkata"

PERSONALIZED_TIME_SLOTS = {
    "Morning": "07:00",
    "Afternoon": "13:00",
    "Evening": "18:00"
}

# Check required inputs and scoring function before making API calls.
required_names = [
    "USER_ACTIVITY",
    "USER_HEALTH_CONDITIONS",
    "USER_TARGET_DATE",
    "score_conditions"
]

missing_names = [
    name for name in required_names
    if name not in globals()
]

if missing_names:
    raise NameError(
        "Missing required definitions: "
        + ", ".join(missing_names)
        + ". Run Step 13A and the cell defining score_conditions() first."
    )

# Validate the date format.
try:
    datetime.strptime(USER_TARGET_DATE, "%Y-%m-%d")
except ValueError:
    raise ValueError("USER_TARGET_DATE must use YYYY-MM-DD format.")

session = requests.Session()

def personalized_get_coordinates(city):
    """Find a city's coordinates, preferring matches in India."""
    response = session.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={
            "name": city,
            "count": 10,
            "language": "en",
            "format": "json"
        },
        timeout=30
    )
    response.raise_for_status()

    matches = response.json().get("results", [])
    if not matches:
        raise ValueError(f"No coordinates found for {city}")

    india_matches = [
        place for place in matches
        if place.get("country_code") == "IN"
    ]

    place = india_matches[0] if india_matches else matches[0]

    return {
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "city": place.get("name", city),
        "state": place.get("admin1", "India")
    }


def personalized_hour_index(times, target_hour, city, data_type):
    """Match an hour against API timestamps such as YYYY-MM-DDTHH:MM."""
    target_timestamp = f"{USER_TARGET_DATE}T{target_hour}"

    for index, timestamp in enumerate(times):
        if timestamp == target_timestamp:
            return index

    # Include available timestamps in the error for diagnosis.
    raise ValueError(
        f"{data_type} forecast unavailable for {city} at "
        f"{target_timestamp}. Sample available timestamps: {times[:3]}"
    )


def personalized_hourly_value(hourly, key, index):
    values = hourly.get(key, [])
    if index >= len(values):
        return None
    return values[index]


def compare_city_personalized(city, period, target_hour):
    """Fetch forecasts and score them using the user's preferences."""

    coordinates = personalized_get_coordinates(city)
    lat = coordinates["latitude"]
    lon = coordinates["longitude"]

    # Weather forecast for the selected date.
    weather_response = session.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "hourly": (
                "temperature_2m,apparent_temperature,"
                "precipitation_probability,precipitation,"
                "relative_humidity_2m,wind_speed_10m,weather_code"
            ),
            "timezone": PERSONALIZED_TIMEZONE,
            "start_date": USER_TARGET_DATE,
            "end_date": USER_TARGET_DATE
        },
        timeout=30
    )
    weather_response.raise_for_status()
    weather_payload = weather_response.json()

    weather_hourly = weather_payload.get("hourly", {})
    weather_times = weather_hourly.get("time", [])

    weather_idx = personalized_hour_index(
        weather_times, target_hour, city, "Weather"
    )

    weather = {
        "temperature": personalized_hourly_value(
            weather_hourly, "temperature_2m", weather_idx
        ),
        "feels_like": personalized_hourly_value(
            weather_hourly, "apparent_temperature", weather_idx
        ),
        "precipitation_probability": personalized_hourly_value(
            weather_hourly, "precipitation_probability", weather_idx
        ),
        "precipitation": personalized_hourly_value(
            weather_hourly, "precipitation", weather_idx
        ),
        "humidity": personalized_hourly_value(
            weather_hourly, "relative_humidity_2m", weather_idx
        ),
        "wind_speed": personalized_hourly_value(
            weather_hourly, "wind_speed_10m", weather_idx
        ),
        "weather_code": personalized_hourly_value(
            weather_hourly, "weather_code", weather_idx
        )
    }

    # Air-quality forecast for the same location, date, and hour.
    air_response = session.get(
        "https://air-quality-api.open-meteo.com/v1/air-quality",
        params={
            "latitude": lat,
            "longitude": lon,
            "hourly": (
                "pm2_5,pm10,us_aqi,european_aqi,"
                "ozone,nitrogen_dioxide"
            ),
            "timezone": PERSONALIZED_TIMEZONE,
            "start_date": USER_TARGET_DATE,
            "end_date": USER_TARGET_DATE
        },
        timeout=30
    )
    air_response.raise_for_status()
    air_payload = air_response.json()

    air_hourly = air_payload.get("hourly", {})
    air_times = air_hourly.get("time", [])

    air_idx = personalized_hour_index(
        air_times, target_hour, city, "Air-quality"
    )

    air_quality = {
        "pm2_5": personalized_hourly_value(
            air_hourly, "pm2_5", air_idx
        ),
        "pm10": personalized_hourly_value(
            air_hourly, "pm10", air_idx
        ),
        "us_aqi": personalized_hourly_value(
            air_hourly, "us_aqi", air_idx
        ),
        "european_aqi": personalized_hourly_value(
            air_hourly, "european_aqi", air_idx
        ),
        "ozone": personalized_hourly_value(
            air_hourly, "ozone", air_idx
        ),
        "nitrogen_dioxide": personalized_hourly_value(
            air_hourly, "nitrogen_dioxide", air_idx
        )
    }

    # Reuse the project's existing scoring logic.
    score, warnings = score_conditions(
        USER_ACTIVITY,
        weather,
        air_quality,
        USER_HEALTH_CONDITIONS
    )

    # Keep location metadata and data separate from old results.
    missing_data = [
        key for key, value in {
            **weather, **air_quality
        }.items()
        if value is None
    ]

    return {
        "location": coordinates["city"],
        "state": coordinates["state"],
        "latitude": lat,
        "longitude": lon,
        "period": period,
        "forecast_time": f"{USER_TARGET_DATE}T{target_hour}",
        "temperature": weather["temperature"],
        "temperature_c": weather["temperature"],
        "pm2_5": air_quality["pm2_5"],
        "us_aqi": air_quality["us_aqi"],
        "score": score,
        "warnings": warnings or [],
        "missing_data": missing_data,
        "activity": USER_ACTIVITY,
        "health_conditions": USER_HEALTH_CONDITIONS
    }


# Run all 9 cities x 3 periods.
personalized_results = []
personalized_unavailable = []

print("Starting personalized GEO AI comparisons...")
print("Date:", USER_TARGET_DATE)
print("Activity:", USER_ACTIVITY)
print("Health considerations:", USER_HEALTH_CONDITIONS or "None")
print()

for city in PERSONALIZED_CITIES:
    for period, target_hour in PERSONALIZED_TIME_SLOTS.items():
        try:
            item = compare_city_personalized(
                city, period, target_hour
            )
            personalized_results.append(item)
            print(
                f"✓ {city} — {period}: "
                f"{item['score']}/100"
            )
        except Exception as error:
            personalized_unavailable.append({
                "city": city,
                "period": period,
                "error": str(error)
            })
            print(f"✗ {city} — {period}: {error}")

personalized_results.sort(
    key=lambda item: item["score"],
    reverse=True
)

print("\n" + "=" * 58)
print("GEO AI — PERSONALIZED COMPARISON SUMMARY")
print("=" * 58)
print("Activity:", USER_ACTIVITY)
print("Health considerations:", USER_HEALTH_CONDITIONS or "None")
print("Forecast date:", USER_TARGET_DATE)
print(
    f"Successful comparisons: {len(personalized_results)}"
    f" / {len(PERSONALIZED_CITIES) * len(PERSONALIZED_TIME_SLOTS)}"
)
print("Unavailable comparisons:", len(personalized_unavailable))

print("\nTOP 5 PERSONALIZED RESULTS")

for rank, item in enumerate(personalized_results[:5], start=1):
    print(
        f"\n#{rank} {item['location']}, {item['state']} "
        f"— {item['period']}"
    )
    print("Forecast:", item["forecast_time"])
    print("Activity:", item["activity"])
    print("Temperature:", item["temperature_c"], "°C")
    print("PM2.5:", item["pm2_5"], "µg/m³")
    print("US AQI:", item["us_aqi"])
    print("Application score:", f"{item['score']}/100")
    print("Warnings:", item["warnings"] or "None generated")
    if item["missing_data"]:
        print("Missing data:", item["missing_data"])

if personalized_unavailable:
    print("\nUNAVAILABLE COMPARISONS")
    for item in personalized_unavailable:
        print(
            f"- {item['city']} / {item['period']}: "
            f"{item['error']}"
        )

if not personalized_results:
    print(
        "\nNo recommendations could be generated. "
        "Check the errors above before relying on results."
    )
elif len(personalized_results) < 27:
    print(
        "\nSome comparisons failed. Use only the successful results "
        "and investigate unavailable entries before presenting the demo."
    )
else:
    print(
        "\nAll 27 comparisons completed. Forecasts can change; "
        "scores are not guarantees of safety."
    )


Starting personalized GEO AI comparisons...
Date: 2026-10-23
Activity: outdoor walking
Health considerations: ['asthma']

✗ Pune — Morning: too many values to unpack (expected 2)
✗ Pune — Afternoon: too many values to unpack (expected 2)
✗ Pune — Evening: too many values to unpack (expected 2)
✗ Mumbai — Morning: too many values to unpack (expected 2)
✗ Mumbai — Afternoon: too many values to unpack (expected 2)
✗ Mumbai — Evening: too many values to unpack (expected 2)
✗ Hyderabad — Morning: too many values to unpack (expected 2)
✗ Hyderabad — Afternoon: too many values to unpack (expected 2)
✗ Hyderabad — Evening: too many values to unpack (expected 2)
✗ Bengaluru — Morning: too many values to unpack (expected 2)
✗ Bengaluru — Afternoon: too many values to unpack (expected 2)
✗ Bengaluru — Evening: too many values to unpack (expected 2)
✗ Kolkata — Morning: too many values to unpack (expected 2)
✗ Kolkata — Afternoon: too many values to unpack (expected 2)
✗ Kolkata — Evening: too man

In [58]:
# GEO AI — find the source of the unpacking error
import inspect

print("Checking get_coordinates...")

try:
    result = get_coordinates("Pune")
    print("Returned value:", repr(result))
    print("Number of returned values:", len(result))
except Exception as e:
    print("get_coordinates error:", repr(e))

print("\nChecking coordinate helper source...")
try:
    print(inspect.getsource(get_coordinates))
except Exception:
    print("Source unavailable; the function may be defined in another cell.")

Checking get_coordinates...
Returned value: (18.51957, 73.85535)
Number of returned values: 2

Checking coordinate helper source...
def get_coordinates(city):
    """Return (latitude, longitude) for a city using Open-Meteo geocoding."""
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": city,
        "count": 10,
        "language": "en",
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    matches = data.get("results", [])
    if not matches:
        raise ValueError(f"No coordinates found for {city}")

    # Prefer a match in India to avoid similarly named cities elsewhere.
    india_matches = [
        place for place in matches
        if place.get("country_code") == "IN"
    ]
    place = india_matches[0] if india_matches else matches[0]

    return place["latitude"], place["longitude"]



In [49]:
# Step 11B: Diagnose LLM generation

print("Model loaded:", "llm" in globals())
print("Results available:", len(multi_time_results))
print("Time slots:", list(TIME_SLOTS.keys()))

test_item = best_by_period.get("Morning")

if test_item:
    print("\nTesting one recommendation...")
    explanation = generate_llm_explanation(test_item)
    print("\nGenerated explanation:\n")
    print(explanation)
else:
    print("No morning recommendation found.")

Model loaded: True
Results available: 27
Time slots: ['Morning', 'Afternoon', 'Evening']

Testing one recommendation...

Generated explanation:

- Location: Bengaluru, Karnataka
- Activity: Outdoor walking
- Time: 2026-10-22T07:00 (Morning)
- Temperature: 21.5 C
- PM2.5: 7.3 µg/m³
- US AQI: 53
- Score: 95/100
- Warnings: Check the precipitation forecast.
- Missing data: []


In [50]:
# Step 11C: Test short CPU inference

response = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": (
                "Bengaluru: 21.2 C, PM2.5 10.7, "
                "US AQI 43. Give one short factual sentence. "
                "Do not invent information."
            )
        }
    ],
    temperature=0.1,
    max_tokens=60
)

print(response["choices"][0]["message"]["content"])

Bengaluru, located in Karnataka, India, recorded a temperature of 21.2°C (70.2°F) on the day, with a PM2.5 level of 10.7 and a US AQI of 43.


In [51]:

# Step 8B: Generate evidence-based recommendation explanations

print("GEO AI — RECOMMENDATION EXPLANATIONS")
print("=" * 60)

if not multi_time_results:
    print("No results available. Run Step 8A first.")
else:
    # Show the top 3 city-time combinations
    top_options = multi_time_results[:3]

    for rank, item in enumerate(top_options, start=1):
        print(f"\n#{rank}: {item['location']}, {item['state']}")
        print(f"Best time: {item['period']} ({item['forecast_time']})")
        print(f"Activity: {ACTIVITY}")
        print(f"Score: {item['score']}/100")

        reasons = []

        temp = item["temperature_c"]
        pm25 = item["pm2_5"]
        aqi = item["us_aqi"]

        if temp is not None:
            if temp <= 25:
                reasons.append(
                    f"The forecast temperature is relatively mild at {temp} °C."
                )
            elif temp >= 35:
                reasons.append(
                    f"The forecast is hot at {temp} °C; consider heat exposure."
                )
            else:
                reasons.append(
                    f"The forecast temperature is {temp} °C."
                )

        if pm25 is not None:
            if pm25 <= 15:
                reasons.append(
                    f"Forecast PM2.5 is comparatively low at {pm25} µg/m³."
                )
            elif pm25 >= 55:
                reasons.append(
                    f"Forecast PM2.5 is elevated at {pm25} µg/m³."
                )
            else:
                reasons.append(
                    f"Forecast PM2.5 is {pm25} µg/m³."
                )

        if aqi is not None:
            reasons.append(f"The forecast US AQI is {aqi}.")

        if item["warnings"]:
            reasons.append(
                "Warnings: " + "; ".join(item["warnings"])
            )
        else:
            reasons.append(
                "The current scoring rules generated no warnings."
            )

        for reason in reasons:
            print("•", reason)

        if item["missing_data"]:
            print("• Missing data:", item["missing_data"])

        print(
            "Note: This is a forecast-based ranking, "
            "not a guarantee of safety."
        )

print("\nExplanation generation complete.")


GEO AI — RECOMMENDATION EXPLANATIONS

#1: Bengaluru, Karnataka
Best time: Morning (2026-10-22T07:00)
Activity: outdoor walking
Score: 95/100
• The forecast temperature is relatively mild at 21.5 °C.
• Forecast PM2.5 is comparatively low at 7.3 µg/m³.
• The forecast US AQI is 53.
• Warnings: Check the precipitation forecast.
Note: This is a forecast-based ranking, not a guarantee of safety.

#2: Pune, Maharashtra
Best time: Morning (2026-10-22T07:00)
Activity: outdoor walking
Score: 90/100
• The forecast temperature is relatively mild at 23.7 °C.
• Forecast PM2.5 is 27.9 µg/m³.
• The forecast US AQI is 101.
• Warnings: Consider air-quality conditions.; Asthma reported: review air quality and personal advice.
Note: This is a forecast-based ranking, not a guarantee of safety.

#3: Hyderabad, Telangana
Best time: Evening (2026-10-22T18:00)
Activity: outdoor walking
Score: 90/100
• The forecast temperature is 29.5 °C.
• Forecast PM2.5 is 34.5 µg/m³.
• The forecast US AQI is 142.
• Warnings:

In [52]:
# Step 9A: Check the environment for LLM integration

import torch
import transformers

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. We can consider a lighter CPU-compatible setup.")

PyTorch version: 2.11.0+cpu
Transformers version: 5.20.2
GPU available: False
No GPU detected. We can consider a lighter CPU-compatible setup.


In [53]:

# Step 8: Interactive Geo AI comparison map

%pip install folium -q


In [54]:

import folium
from IPython.display import display

if not results:
    raise ValueError("No comparison results available to map.")

# Center the map around the compared locations
center_lat = sum(r["latitude"] for r in results) / len(results)
center_lon = sum(r["longitude"] for r in results) / len(results)



geo_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=5,
    tiles="Esri.WorldGrayCanvas"
)



for item in results:
    score = item["score"]

    if score >= 80:
        color = "green"
    elif score >= 60:
        color = "orange"
    else:
        color = "red"

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; width: 280px; line-height: 1.6;">
        <h3 style="margin: 0 0 8px 0;">
            📍 {item['location']}, {item['state']}
        </h3>
        <div style="font-size: 20px; font-weight: bold;">
            ⭐ Score: {score}/100
        </div>
        <hr>
        <b>🕒 Forecast:</b> {item['forecast_time']}<br>
        <b>🌡️ Temperature:</b> {item['temperature_c']} °C<br>
        <b>🌫️ PM2.5:</b> {item['pm2_5']} µg/m³<br>
        <b>💨 US AQI:</b> {item['us_aqi']}<br>
        <hr>
        <b>⚠️ Warnings:</b><br>
        {', '.join(item['warnings']) if item['warnings'] else 'None'}
    </div>
    """

    folium.Marker(
        location=[item["latitude"], item["longitude"]],
        tooltip=f"{item['location']} — {score}/100",
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(color=color, icon="info-sign")
    ).add_to(geo_map)

display(geo_map)

In [59]:
# GEO AI — find the source of the unpacking error
import inspect

print("Checking get_coordinates...")

try:
    result = get_coordinates("Pune")
    print("Returned value:", repr(result))
    print("Number of returned values:", len(result))
except Exception as e:
    print("get_coordinates error:", repr(e))

print("\nChecking coordinate helper source...")
try:
    print(inspect.getsource(get_coordinates))
except Exception:
    print("Source unavailable; the function may be defined in another cell.")

Checking get_coordinates...
Returned value: (18.51957, 73.85535)
Number of returned values: 2

Checking coordinate helper source...
def get_coordinates(city):
    """Return (latitude, longitude) for a city using Open-Meteo geocoding."""
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": city,
        "count": 10,
        "language": "en",
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    matches = data.get("results", [])
    if not matches:
        raise ValueError(f"No coordinates found for {city}")

    # Prefer a match in India to avoid similarly named cities elsewhere.
    india_matches = [
        place for place in matches
        if place.get("country_code") == "IN"
    ]
    place = india_matches[0] if india_matches else matches[0]

    return place["latitude"], place["longitude"]



In [4]:

!pip -q install geopy


In [10]:

import requests
import pandas as pd
import json
from geopy.distance import geodesic
from datetime import datetime, timezone

print("GEO AI Member 2 integration initialized!")


GEO AI Member 2 integration initialized!


In [11]:

# GEO AI — Member 2 live-data functions
# Kept separate from Member 1's working recommendation engine.

def m2_get_coordinates(place_name):
    url = "https://nominatim.openstreetmap.org/search"

    params = {
        "q": place_name,
        "format": "jsonv2",
        "limit": 1
    }

    headers = {
        "User-Agent": "GeoAI-Hackathon-StudentProject/1.0"
    }

    response = requests.get(
        url, params=params, headers=headers, timeout=20
    )
    response.raise_for_status()
    results = response.json()

    if not results:
        raise ValueError(f"Location not found: {place_name}")

    result = results[0]

    return {
        "name": result["display_name"],
        "latitude": float(result["lat"]),
        "longitude": float(result["lon"])
    }


def m2_get_weather(latitude, longitude, forecast_days=2):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": (
            "temperature_2m,"
            "precipitation_probability,"
            "precipitation,"
            "wind_speed_10m,"
            "wind_direction_10m,"
            "cloud_cover"
        ),
        "forecast_days": forecast_days,
        "timezone": "auto"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    weather = pd.DataFrame(data["hourly"])
    weather["time"] = pd.to_datetime(weather["time"])

    return weather, data.get("hourly_units", {})


def m2_get_air_quality(latitude, longitude, forecast_days=2):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "us_aqi,pm2_5,pm10",
        "forecast_days": forecast_days,
        "timezone": "auto"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    air = pd.DataFrame(data["hourly"])
    air["time"] = pd.to_datetime(air["time"])

    return air, data.get("hourly_units", {})


def m2_get_nearby_places(latitude, longitude, radius_km=5):
    radius_m = int(radius_km * 1000)

    query = f"""
    [out:json][timeout:15];
    (
      node(around:{radius_m},{latitude},{longitude})["leisure"~"park|garden|pitch|sports_centre|track"];
      way(around:{radius_m},{latitude},{longitude})["leisure"~"park|garden|pitch|sports_centre|track"];
      relation(around:{radius_m},{latitude},{longitude})["leisure"~"park|garden|pitch|sports_centre|track"];
      node(around:{radius_m},{latitude},{longitude})["natural"="beach"];
      way(around:{radius_m},{latitude},{longitude})["natural"="beach"];
    );
    out center tags;
    """

    endpoints = [
        "https://overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter"
    ]

    headers = {
        "User-Agent": "GeoAI-Hackathon-StudentProject/1.0",
        "Accept": "application/json"
    }

    elements = None

    for endpoint in endpoints:
        try:
            response = requests.post(
                endpoint,
                data={"data": query},
                headers=headers,
                timeout=20
            )
            response.raise_for_status()
            elements = response.json().get("elements", [])
            break
        except (requests.RequestException, ValueError) as e:
            print(f"Nearby-place endpoint failed: {e}")

    if elements is None:
        print("Warning: Nearby places unavailable; continuing.")
        return pd.DataFrame(columns=[
            "name", "type", "latitude", "longitude", "distance_km"
        ])

    places = []

    for element in elements:
        tags = element.get("tags", {})

        if "lat" in element and "lon" in element:
            lat, lon = element["lat"], element["lon"]
        elif "center" in element:
            lat = element["center"]["lat"]
            lon = element["center"]["lon"]
        else:
            continue

        places.append({
            "name": tags.get("name", "Unnamed place"),
            "type": tags.get(
                "leisure", tags.get("natural", "outdoor")
            ),
            "latitude": lat,
            "longitude": lon,
            "distance_km": round(
                geodesic((latitude, longitude), (lat, lon)).km, 2
            )
        })

    places_df = pd.DataFrame(places)

    if not places_df.empty:
        places_df = places_df.sort_values(
            "distance_km"
        ).reset_index(drop=True)

    return places_df


def m2_get_live_data(latitude, longitude, radius_km=5):
    weather, weather_units = m2_get_weather(latitude, longitude)
    air, air_units = m2_get_air_quality(latitude, longitude)

    places = m2_get_nearby_places(
        latitude, longitude, radius_km
    )

    return {
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "location": {
            "latitude": latitude,
            "longitude": longitude
        },
        "weather": {
            "data": weather.to_dict(orient="records"),
            "units": weather_units
        },
        "air_quality": {
            "data": air.to_dict(orient="records"),
            "units": air_units
        },
        "nearby_places": places.to_dict(orient="records")
    }


print("All five Member 2 functions are ready!")


All five Member 2 functions are ready!


In [13]:

# GEO AI — Integration test

selected_place = m2_get_coordinates(
    "Hyderabad, Telangana, India"
)

print("Location found:", selected_place["name"])
print(
    "Coordinates:",
    selected_place["latitude"],
    selected_place["longitude"]
)

member2_data = m2_get_live_data(
    selected_place["latitude"],
    selected_place["longitude"],
    radius_km=5
)

weather_df_m2 = pd.DataFrame(
    member2_data["weather"]["data"]
)

air_df_m2 = pd.DataFrame(
    member2_data["air_quality"]["data"]
)

places_df_m2 = pd.DataFrame(
    member2_data["nearby_places"]
)

print("\n--- MEMBER 2 INTEGRATION RESULTS ---")
print("Weather forecast records:", len(weather_df_m2))
print("Air-quality forecast records:", len(air_df_m2))
print("Nearby places found:", len(places_df_m2))

print("\nWeather sample:")
display(weather_df_m2.head(3))

print("\nAir-quality sample:")
display(air_df_m2.head(3))

print("\nNearby places sample:")
display(places_df_m2.head(5))


Location found: Hyderabad, Bahadurpura mandal, Hyderabad, Telangana, India
Coordinates: 17.360589 78.4740613

--- MEMBER 2 INTEGRATION RESULTS ---
Weather forecast records: 48
Air-quality forecast records: 48
Nearby places found: 256

Weather sample:


,time,temperature_2m,precipitation_probability,precipitation,wind_speed_10m,wind_direction_10m,cloud_cover
0,2026-10-24 00:00:00,26.0,0,0.0,5.7,77,39
1,2026-10-24 01:00:00,25.6,0,0.0,5.1,42,37
2,2026-10-24 02:00:00,25.3,0,0.0,4.9,33,30



Air-quality sample:


,time,us_aqi,pm2_5,pm10
0,2026-10-24 00:00:00,97,41.9,44.9
1,2026-10-24 01:00:00,97,44.6,47.8
2,2026-10-24 02:00:00,98,47.0,50.6



Nearby places sample:


,name,type,latitude,longitude,distance_km
0,Unnamed place,park,17.358771,78.471753,0.32
1,Unnamed place,park,17.361812,78.470926,0.36
2,Unnamed place,park,17.360013,78.477938,0.42
3,Unnamed place,park,17.357315,78.471600,0.45
4,Unnamed place,park,17.356431,78.471498,0.53


In [14]:

# Save Member 2's live-data output
with open("geoai_member2_live_data.json", "w") as f:
    json.dump(member2_data, f, indent=2, default=str)

print("Member 2 live data saved successfully!")


Member 2 live data saved successfully!


In [15]:

print("Results type:", type(results))

if isinstance(results, list) and results:
    print("First result:", results[0])
elif isinstance(results, pd.DataFrame):
    print("Columns:", results.columns.tolist())
    display(results.head(2))


Results type: <class 'list'>
First result: {'city': 'Hyderabad', 'state': 'Telangana', 'period': 'Morning', 'time': '2026-10-25T07:00', 'temp': 25.0, 'rain': 0, 'pm25': 28.9, 'aqi': 100, 'score': 90, 'warnings': ['Consider PM2.5', 'AQI above Good', 'Asthma: review air quality']}


In [18]:
# GEO AI — Connect Recommendation Engine to Member 2 Live Data

# Get the top-ranked recommendation from your existing results list
best_result = results[0]

recommended_city = best_result["city"]
recommended_state = best_result["state"]

print("GEO AI — INTEGRATED RECOMMENDATION")
print("=" * 45)
print(f"Recommended location: {recommended_city}, {recommended_state}")
print(f"Activity: outdoor walking")
print(f"Recommended period: {best_result['period']}")
print(f"Recommended time: {best_result['time']}")
print(f"Application score: {best_result['score']}/100")
print(f"Temperature: {best_result['temp']} °C")
print(f"PM2.5: {best_result['pm25']} µg/m³")
print(f"US AQI: {best_result['aqi']}")
print(f"Warnings: {best_result['warnings']}")

print("\nConnecting to Member 2 live-data functions...")

# Get coordinates for the recommended city
recommended_location = m2_get_coordinates(
    f"{recommended_city}, {recommended_state}, India"
)

print("\nLocation coordinates:")
print("Latitude:", recommended_location["latitude"])
print("Longitude:", recommended_location["longitude"])

# Fetch live forecast and nearby-place context
integrated_data = m2_get_live_data(
    recommended_location["latitude"],
    recommended_location["longitude"],
    radius_km=5
)

print("\n--- MEMBER 2 LIVE DATA SUMMARY ---")
print("Location:", recommended_location.get("name", f"{recommended_city}, {recommended_state}"))
print(
    "Weather records:",
    len(integrated_data["weather"]["data"])
)
print(
    "Air-quality records:",
    len(integrated_data["air_quality"]["data"])
)
print(
    "Nearby places:",
    len(integrated_data["nearby_places"])
)

print("\nIntegration completed.")

GEO AI — INTEGRATED RECOMMENDATION
Recommended location: Hyderabad, Telangana
Activity: outdoor walking
Recommended period: Morning
Recommended time: 2026-10-25T07:00
Application score: 90/100
Temperature: 25.0 °C
PM2.5: 28.9 µg/m³
US AQI: 100
Warnings: ['Consider PM2.5', 'AQI above Good', 'Asthma: review air quality']

Connecting to Member 2 live-data functions...

Location coordinates:
Latitude: 17.360589
Longitude: 78.4740613
Nearby-place endpoint failed: HTTPSConnectionPool(host='overpass-api.de', port=443): Read timed out. (read timeout=20)
Nearby-place endpoint failed: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=20)

--- MEMBER 2 LIVE DATA SUMMARY ---
Location: Hyderabad, Bahadurpura mandal, Hyderabad, Telangana, India
Weather records: 48
Air-quality records: 48
Nearby places: 0

Integration completed.


In [19]:
# GEO AI — Safe Integration Summary

print("\n--- MEMBER 2 LIVE DATA SUMMARY ---")

print(
    "Location:",
    recommended_location.get(
        "name",
        f"{recommended_city}, {recommended_state}"
    )
)

print(
    "Weather records:",
    len(integrated_data.get("weather", {}).get("data", []))
)

print(
    "Air-quality records:",
    len(integrated_data.get("air_quality", {}).get("data", []))
)

print(
    "Nearby places:",
    len(integrated_data.get("nearby_places", []))
)

print("\nIntegration summary completed!")


--- MEMBER 2 LIVE DATA SUMMARY ---
Location: Hyderabad, Bahadurpura mandal, Hyderabad, Telangana, India
Weather records: 48
Air-quality records: 48
Nearby places: 0

Integration summary completed!


In [20]:
import json

final_geoai_output = {
    "recommendation": best_result,
    "location": recommended_location,
    "live_data": integrated_data
}

with open("geoai_integrated_output.json", "w") as f:
    json.dump(final_geoai_output, f, indent=2, default=str)

print("Saved: geoai_integrated_output.json")

Saved: geoai_integrated_output.json


In [21]:
import json

final_geoai_output = {
    "recommendation": best_result,
    "location": recommended_location,
    "live_data": integrated_data
}

with open("geoai_integrated_output.json", "w") as f:
    json.dump(final_geoai_output, f, indent=2, default=str)

print("Saved: geoai_integrated_output.json")

Saved: geoai_integrated_output.json
